In [ ]:
!pip install seaborn
!pip install graphviz
!pip install statsmodels
!pip install xgboost
!pip install lightgbm
!pip install catboost
!pip install mlxtend

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import graphviz

import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
from statsmodels.api import qqplot, add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, GradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, precision_score, r2_score, recall_score, roc_auc_score, silhouette_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_graphviz

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

print("모든 패키지 import 성공")

In [ ]:
df_raw1 = pd.read_csv(r"../data/processed/bat_process.csv", encoding = "euc-kr")
display(df_raw1.head())
df_raw2 = pd.read_csv(r"../data/processed/bat_tat.csv", encoding = "euc-kr")
display(df_raw2.head())

In [ ]:
# df_raw1
df_raw1_na = df_raw1[df_raw1.isnull().any(axis=1)]  
display(df_raw1_na)

# df_raw2
df_raw2_na = df_raw2[df_raw2.isnull().any(axis=1)]
display(df_raw2_na)

In [ ]:
def missing_summary(df):
    summary = pd.DataFrame({
        "missing_count": df.isnull().sum(),
        "missing_ratio(%)": df.isnull().mean() * 100
    })
    summary = summary[summary["missing_count"] > 0]
    return summary.sort_values(by="missing_count", ascending=False)

display(missing_summary(df_raw1))
display(missing_summary(df_raw2))

In [ ]:
def plot_missing(df, title):
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    plt.figure(figsize=(10,5))
    missing.plot(kind='bar')
    plt.title(title)
    plt.ylabel("Missing Count")
    plt.xticks(rotation=45)
    plt.show()

plot_missing(df_raw1, "df_raw1 Missing Count")

In [ ]:
plt.figure(figsize=(10,2))
sns.heatmap(df_raw1.isnull().mean().to_frame().T, annot=True, cmap="coolwarm")
plt.title("df_raw1 Missing Ratio")
plt.show()

In [ ]:
df_raw1.groupby("lot_id")["ocv2_deltaocv"].apply(lambda x: x.isnull().mean())

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 0. 설정
# =========================================================
INPUT_FILE = "bat_process.csv"
OUTPUT_DIR = "outlier_hist_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/histograms", exist_ok=True)

# ---------------------------------------------------------
# 사용자 수정 영역 1) 변수 유형 분류
# ---------------------------------------------------------
TYPE_A_DATA_ERROR = [
    # 물리적으로 말이 안 되는 값이 나오면 제거/NaN 처리할 변수
    "ocv1_ocv", "ocv2_ocv", "socv1_ocv", "socv2_ocv", "socv3_ocv",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "pg1_imp", "pg1_impfit", "pc1_imp", "m1_res_ac",
    "m1_thick",
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
]

TYPE_B_RECIPE = [
    # 설정값/제어값: 삭제하지 않고 spec 이탈 flag
    "c1_curr_end", "dc1_curr_end", "c2_curr_end", "dc2_curr_end",
    "c3_curr_end", "dc3_curr_end", "c4_curr_end",
    "c3_cvval", "c4_cvval",
    "c3_ccval", "c4_ccval",
]

TYPE_C_PROCESS = [
    # 실제 공정 이상 신호: 삭제 금지, flag만
    "ocv2_deltaocv", "pg1_imp", "pc1_imp", "m1_res_ac", "m1_thick",
    "c3_time_cv", "c4_time_cv",
]

TYPE_D_NORMAL_VARIATION = [
    # 자연 편차가 있는 변수: 필요 시 clip
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "c1_voltage_avg", "dc1_voltage_avg",
    "c2_voltage_avg", "dc2_voltage_avg",
    "c3_voltage_avg", "dc3_voltage_avg",
    "c4_voltage_avg",
]

# ---------------------------------------------------------
# 사용자 수정 영역 2) 하드 리밋 / 정상범위
# 단위는 네 데이터 기준으로 맞춰야 함
# 예시는 현재 데이터 스케일을 고려한 대략적 sanity check 예시
# ---------------------------------------------------------
HARD_LIMITS = {
    # A. 데이터 오류형: 이 범위를 벗어나면 NaN 처리
    # 전압 계열 (예: mV로 보이는 경우)
    "ocv1_ocv": (0, 5000),
    "ocv2_ocv": (0, 5000),
    "socv1_ocv": (0, 5000),
    "socv2_ocv": (0, 5000),
    "socv3_ocv": (0, 5000),

    # 시간 계열
    "c1_time_cc": (0, 100000),
    "c2_time_cc": (0, 100000),
    "c3_time_cc": (0, 100000),
    "c4_time_cc": (0, 100000),
    "c3_time_cv": (0, 100000),
    "c4_time_cv": (0, 100000),

    # 임피던스/저항
    "pg1_imp": (0, 10000),
    "pg1_impfit": (0, 10000),
    "pc1_imp": (0, 10000),
    "m1_res_ac": (0, 10000),

    # 두께
    "m1_thick": (0, 10000),

    # 온도 계열 (예: 0.1도 스케일이면 200~400 정도일 수 있음)
    "c1_temp_avg": (0, 1000),
    "dc1_temp_avg": (0, 1000),
    "c2_temp_avg": (0, 1000),
    "dc2_temp_avg": (0, 1000),
    "c3_temp_avg": (0, 1000),
    "dc3_temp_avg": (0, 1000),
    "c4_temp_avg": (0, 1000),
}

SPEC_LIMITS = {
    # B. 레시피 이탈형: flag만 생성
    # 실제 단위에 맞게 수정 필요
    "c1_curr_end": (0, 10000),
    "dc1_curr_end": (0, 10000),
    "c2_curr_end": (0, 10000),
    "dc2_curr_end": (0, 10000),
    "c3_curr_end": (0, 10000),
    "dc3_curr_end": (0, 10000),
    "c4_curr_end": (0, 10000),
    "c3_cvval": (3500, 4500),
    "c4_cvval": (3500, 4500),
    "c3_ccval": (0, 10000),
    "c4_ccval": (0, 10000),
}

PROCESS_LIMITS = {
    # C. 공정 이상형: flag만 생성
    # 절대 삭제 금지, 위험 신호로 보존
    "ocv2_deltaocv": (None, 50),   # 상한 예시
    "pg1_imp": (None, 500),        # 상한 예시
    "pc1_imp": (None, 500),        # 상한 예시
    "m1_res_ac": (None, 500),      # 상한 예시
    "m1_thick": (None, 5000),      # 상한 예시
    "c3_time_cv": (None, 50000),   # 상한 예시
    "c4_time_cv": (None, 50000),   # 상한 예시
}

# =========================================================
# 1. 유틸 함수
# =========================================================
def iqr_clip(series: pd.Series, k: float = 1.5):
    s = series.dropna()
    if len(s) == 0:
        return series.copy(), np.nan, np.nan
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    clipped = series.clip(lower=low, upper=high)
    return clipped, low, high

def apply_hard_limit(series: pd.Series, low=None, high=None):
    s = series.copy()
    if low is not None:
        s = s.mask(s < low, np.nan)
    if high is not None:
        s = s.mask(s > high, np.nan)
    return s

def make_flag(series: pd.Series, low=None, high=None):
    flag = pd.Series(0, index=series.index, dtype="int64")
    if low is not None:
        flag = flag | (series < low)
    if high is not None:
        flag = flag | (series > high)
    return flag.astype(int)

def plot_before_after_hist(before: pd.Series, after: pd.Series, col: str, out_path: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(before.dropna(), bins=40, alpha=0.8)
    axes[0].set_title(f"{col} - Before")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Count")

    axes[1].hist(after.dropna(), bins=40, alpha=0.8)
    axes[1].set_title(f"{col} - After")
    axes[1].set_xlabel(col)
    axes[1].set_ylabel("Count")

    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

# =========================================================
# 2. 데이터 로드
# =========================================================
df = pd.read_csv(INPUT_FILE, encoding="euc-kr")
df_before = df.copy()
df_after = df.copy()

summary_rows = []

# =========================================================
# 3. A. 데이터 오류형 처리
#    - 물리적으로 불가능한 값: NaN 처리
# =========================================================
for col in TYPE_A_DATA_ERROR:
    if col not in df_after.columns:
        continue

    before = df_after[col].copy()
    low, high = HARD_LIMITS.get(col, (None, None))

    after = apply_hard_limit(before, low=low, high=high)
    df_after[col] = after

    changed = before.notna().sum() - after.notna().sum()

    summary_rows.append({
        "column": col,
        "type": "A_data_error",
        "action": "hard_limit_to_nan",
        "low": low,
        "high": high,
        "changed_count": int(changed)
    })

    plot_before_after_hist(
        before,
        after,
        col,
        f"{OUTPUT_DIR}/histograms/{col}_A_before_after.png"
    )

# =========================================================
# 4. B. 레시피 이탈형 처리
#    - 삭제 X, flag만 생성
# =========================================================
for col in TYPE_B_RECIPE:
    if col not in df_after.columns:
        continue

    before = df_after[col].copy()
    low, high = SPEC_LIMITS.get(col, (None, None))

    flag_col = f"{col}_flag_spec"
    df_after[flag_col] = make_flag(before, low=low, high=high)

    # 값 자체는 유지
    after = before.copy()

    summary_rows.append({
        "column": col,
        "type": "B_recipe_deviation",
        "action": "flag_only",
        "low": low,
        "high": high,
        "changed_count": int(df_after[flag_col].sum())
    })

    plot_before_after_hist(
        before,
        after,
        col,
        f"{OUTPUT_DIR}/histograms/{col}_B_before_after.png"
    )

# =========================================================
# 5. C. 공정 이상형 처리
#    - 삭제 X, flag만 생성
# =========================================================
for col in TYPE_C_PROCESS:
    if col not in df_after.columns:
        continue

    before = df_after[col].copy()
    low, high = PROCESS_LIMITS.get(col, (None, None))

    flag_col = f"{col}_flag_process"
    df_after[flag_col] = make_flag(before, low=low, high=high)

    # 값 자체는 유지
    after = before.copy()

    summary_rows.append({
        "column": col,
        "type": "C_process_anomaly",
        "action": "flag_only",
        "low": low,
        "high": high,
        "changed_count": int(df_after[flag_col].sum())
    })

    plot_before_after_hist(
        before,
        after,
        col,
        f"{OUTPUT_DIR}/histograms/{col}_C_before_after.png"
    )

# =========================================================
# 6. D. 정상 변동형 처리
#    - 삭제보다는 IQR clip
# =========================================================
for col in TYPE_D_NORMAL_VARIATION:
    if col not in df_after.columns:
        continue

    before = df_after[col].copy()

    # 이미 A 처리에서 NaN이 생겼을 수 있으니 그 상태 기준으로 clip
    after, low, high = iqr_clip(before, k=1.5)
    df_after[col] = after

    changed = (before != after).sum(skipna=True)

    summary_rows.append({
        "column": col,
        "type": "D_normal_variation",
        "action": "iqr_clip",
        "low": low,
        "high": high,
        "changed_count": int(changed)
    })

    plot_before_after_hist(
        before,
        after,
        col,
        f"{OUTPUT_DIR}/histograms/{col}_D_before_after.png"
    )

# =========================================================
# 7. 결과 저장
# =========================================================
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUTPUT_DIR}/outlier_processing_summary.csv", index=False, encoding="utf-8-sig")
df_after.to_csv(f"{OUTPUT_DIR}/bat_process_after_outlier_handling.csv", index=False, encoding="utf-8-sig")

print("완료")
print(f"- 처리 결과: {OUTPUT_DIR}/bat_process_after_outlier_handling.csv")
print(f"- 요약표: {OUTPUT_DIR}/outlier_processing_summary.csv")
print(f"- 히스토그램: {OUTPUT_DIR}/histograms/")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 1. 데이터 로드
# -----------------------------
df = pd.read_csv("bat_process.csv", encoding="euc-kr")

# 폴더 생성
os.makedirs("output_figs/before_after", exist_ok=True)
os.makedirs("output_figs/by_judge", exist_ok=True)

# -----------------------------
# 2. 타깃 인코딩
# -----------------------------
df["judge_bin"] = (df["judge"] == "불량").astype(int)

# -----------------------------
# 3. 파생변수 생성
# -----------------------------
# 3-1. 효율
df["eff_c1"] = df["dc1_capa"] / df["c1_capa"].replace(0, np.nan)
df["eff_c2"] = df["dc2_capa"] / df["c2_capa"].replace(0, np.nan)
df["eff_c3"] = df["dc3_capa"] / df["c3_capa"].replace(0, np.nan)

# 3-2. 시간-전류-용량 일관성
# 주의: c1_capa, c2_capa는 데이터 정의 문제 가능성이 있으므로 일단 c3, c4 위주 권장
df["time_ratio_c3"] = df["c3_time_cc"] / (df["c3_capa"] / df["c3_ccval"].replace(0, np.nan))
df["time_ratio_c4"] = df["c4_time_cc"] / (df["c4_capa"] / df["c4_ccval"].replace(0, np.nan))

# 3-3. CV 비율
if "c3_time_cv" in df.columns:
    df["cv_ratio_c3"] = df["c3_time_cv"] / (df["c3_time_cc"] + df["c3_time_cv"])
if "c4_time_cv" in df.columns:
    df["cv_ratio_c4"] = df["c4_time_cv"] / (df["c4_time_cc"] + df["c4_time_cv"])

# 3-4. 전압 gap
df["vgap_1"] = df["c1_voltage_avg"] - df["dc1_voltage_avg"]
df["vgap_2"] = df["c2_voltage_avg"] - df["dc2_voltage_avg"]
df["vgap_3"] = df["c3_voltage_avg"] - df["dc3_voltage_avg"]

# 3-5. deltaocv 누적
delta_cols = [c for c in ["ocv2_deltaocv"] if c in df.columns]
if delta_cols:
    df["deltaocv_total"] = df[delta_cols].fillna(0).sum(axis=1)

# 3-6. 임피던스 편차 (lot 기준)
for col in ["pg1_imp", "pc1_imp", "m1_res_ac", "m1_thick"]:
    if col in df.columns:
        df[f"{col}_dev_lot"] = df[col] - df.groupby("lot_id")[col].transform("median")

# 3-7. fit 대비 차이
if "pg1_imp" in df.columns and "pg1_impfit" in df.columns:
    df["pg1_imp_gap"] = df["pg1_imp"] - df["pg1_impfit"]
if "dc3_capa" in df.columns and "dc3_capafit" in df.columns:
    df["dc3_capa_gap"] = df["dc3_capa"] - df["dc3_capafit"]

# 3-8. 온도 편차
temp_cols = [c for c in ["c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg", "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg"] if c in df.columns]
for col in temp_cols:
    df[f"{col}_dev"] = df[col] - df[col].median()

# -----------------------------
# 4. 이상치 처리 함수
# -----------------------------
def iqr_clip(series, k=1.5):
    x = series.copy()
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    clipped = x.clip(lower=low, upper=high)
    return clipped, low, high

def plot_before_after(series_before, series_after, name, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # boxplot
    axes[0].boxplot([series_before.dropna(), series_after.dropna()], tick_labels=["Before", "After"])
    axes[0].set_title(f"{name} - Boxplot")

    # histogram
    axes[1].hist(series_before.dropna(), bins=40, alpha=0.6, label="Before")
    axes[1].hist(series_after.dropna(), bins=40, alpha=0.6, label="After")
    axes[1].set_title(f"{name} - Histogram")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{name}_before_after.png"), dpi=150)
    plt.close()

def plot_by_judge(df_in, col, save_dir):
    good = df_in.loc[df_in["judge_bin"] == 0, col].dropna()
    bad = df_in.loc[df_in["judge_bin"] == 1, col].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].boxplot([good, bad], tick_labels=["양품", "불량"])
    axes[0].set_title(f"{col} - 양품/불량 Boxplot")

    axes[1].hist(good, bins=40, alpha=0.6, label="양품")
    axes[1].hist(bad, bins=40, alpha=0.6, label="불량")
    axes[1].set_title(f"{col} - 양품/불량 Histogram")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{col}_by_judge.png"), dpi=150)
    plt.close()

# -----------------------------
# 5. 시각화 대상 변수 선택
# -----------------------------
feature_candidates = [
    "eff_c1", "eff_c2", "eff_c3",
    "time_ratio_c3", "time_ratio_c4",
    "cv_ratio_c3", "cv_ratio_c4",
    "vgap_1", "vgap_2", "vgap_3",
    "deltaocv_total",
    "pg1_imp_dev_lot", "pc1_imp_dev_lot", "m1_res_ac_dev_lot", "m1_thick_dev_lot",
    "pg1_imp_gap", "dc3_capa_gap",
]

feature_candidates += [f"{c}_dev" for c in temp_cols]

feature_candidates = [c for c in feature_candidates if c in df.columns]

# -----------------------------
# 6. 이상치 처리 전/후 데이터 생성
#    주의: 여기서는 "삭제"가 아니라 "시각화용 clip" 예시
# -----------------------------
df_after = df.copy()
summary_rows = []

for col in feature_candidates:
    before = df[col]
    after, low, high = iqr_clip(before, k=1.5)
    df_after[col] = after

    outlier_cnt = ((before < low) | (before > high)).sum()

    summary_rows.append({
        "feature": col,
        "low": low,
        "high": high,
        "outlier_count": int(outlier_cnt),
        "outlier_ratio": round(outlier_cnt / len(df), 4)
    })

    # 처리 전/후 시각화
    plot_before_after(before, after, col, "output_figs/before_after")

    # 양품/불량 비교 시각화(처리 전 기준)
    plot_by_judge(df, col, "output_figs/by_judge")

summary = pd.DataFrame(summary_rows).sort_values("outlier_ratio", ascending=False)
summary.to_csv("output_figs/outlier_summary.csv", index=False, encoding="utf-8-sig")

print("완료")
print("저장된 파일:")
print("- output_figs/outlier_summary.csv")
print("- output_figs/before_after/*.png")
print("- output_figs/by_judge/*.png")

In [ ]:
flag_cols = [c for c in df_after.columns if c.endswith("_flag_spec") or c.endswith("_flag_process")]
flag_summary = pd.DataFrame({
    "flag_col": flag_cols,
    "flag_count": [df_after[c].sum() for c in flag_cols],
    "flag_ratio": [df_after[c].mean() for c in flag_cols]
}).sort_values("flag_ratio", ascending=False)

flag_summary.to_csv(f"{OUTPUT_DIR}/flag_summary.csv", index=False, encoding="utf-8-sig")
print(flag_summary.head(20))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 0. 파일 경로 설정
# =========================================================
SUMMARY_FILE = r"outlier_hist_results\outlier_processing_summary.csv"
BEFORE_FILE = r"bat_process.csv"
AFTER_FILE = r"outlier_hist_results\bat_process_after_outlier_handling.csv"

OUT_DIR = "outlier_effect_report"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/type_summary", exist_ok=True)
os.makedirs(f"{OUT_DIR}/d_type_compare", exist_ok=True)
os.makedirs(f"{OUT_DIR}/bc_type_flags", exist_ok=True)

# =========================================================
# 1. 데이터 로드
# =========================================================
summary = pd.read_csv(SUMMARY_FILE)
df_before = pd.read_csv(BEFORE_FILE, encoding="euc-kr")
df_after = pd.read_csv(AFTER_FILE, encoding="utf-8-sig")

# judge 컬럼 있으면 binary 생성
if "judge" in df_before.columns:
    df_before["judge_bin"] = (df_before["judge"] == "불량").astype(int)
    df_after["judge_bin"] = (df_after["judge"] == "불량").astype(int)

n_rows = len(df_before)

# =========================================================
# 2. 요약표 보강
# =========================================================
summary["changed_ratio"] = summary["changed_count"] / n_rows

# action별 의미 추가
def effect_type(row):
    if row["action"] == "flag_only":
        return "flag_only"
    elif row["action"] in ["iqr_clip", "hard_limit_to_nan"]:
        return "value_changed"
    return "other"

summary["effect_type"] = summary.apply(effect_type, axis=1)

# =========================================================
# 3. 왜 전후 차이가 안 보였는지 설명용 표
# =========================================================
type_overview = (
    summary.groupby(["type", "action", "effect_type"], as_index=False)
    .agg(
        variable_count=("column", "count"),
        total_changed=("changed_count", "sum"),
        mean_changed_ratio=("changed_ratio", "mean"),
        max_changed_ratio=("changed_ratio", "max")
    )
)

type_overview["mean_changed_ratio_pct"] = (type_overview["mean_changed_ratio"] * 100).round(2)
type_overview["max_changed_ratio_pct"] = (type_overview["max_changed_ratio"] * 100).round(2)

def explain_row(row):
    if row["effect_type"] == "flag_only":
        return "값은 유지되고 flag만 생성되어 히스토그램 전후 차이가 거의 보이지 않음"
    if row["action"] == "hard_limit_to_nan" and row["total_changed"] == 0:
        return "하드리밋 위반 값이 없어 실제 변경이 없음"
    if row["action"] == "iqr_clip":
        if row["max_changed_ratio"] < 0.03:
            return "clip 대상 비율이 작아 전체 히스토그램에서는 차이가 미미함"
        elif row["max_changed_ratio"] < 0.10:
            return "일부 tail 값만 clip되어 boxplot/tail 확대에서 더 잘 보임"
        else:
            return "변경 비율이 상대적으로 커서 tail 비교 시 처리 효과 확인 가능"
    return "추가 확인 필요"

type_overview["why_difference_not_visible"] = type_overview.apply(explain_row, axis=1)
type_overview.to_csv(f"{OUT_DIR}/type_summary/type_overview.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 4. 처리 유형별 barplot
# =========================================================
plt.figure(figsize=(12, 5))
x = np.arange(len(type_overview))
plt.bar(x, type_overview["total_changed"])
plt.xticks(x, type_overview["type"] + "\n" + type_overview["action"], rotation=30, ha="right")
plt.ylabel("Total changed count")
plt.title("처리 유형별 실제 변경 건수")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/type_summary/changed_count_by_type.png", dpi=150)
plt.close()

plt.figure(figsize=(12, 5))
plt.bar(x, type_overview["max_changed_ratio_pct"])
plt.xticks(x, type_overview["type"] + "\n" + type_overview["action"], rotation=30, ha="right")
plt.ylabel("Max changed ratio (%)")
plt.title("처리 유형별 최대 변경 비율")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/type_summary/max_changed_ratio_by_type.png", dpi=150)
plt.close()

# =========================================================
# 5. D 유형: 실제 값이 바뀐 변수들 비교 시각화
#    - 전체 hist
#    - tail 확대 hist
#    - boxplot
# =========================================================
d_vars = summary.loc[summary["type"] == "D_normal_variation", "column"].tolist()

def tail_zoom_range(before_s, after_s):
    s = pd.concat([before_s, after_s], axis=0).dropna()
    if len(s) == 0:
        return None, None
    q95 = s.quantile(0.95)
    q995 = s.quantile(0.995)
    return q95, q995

for col in d_vars:
    if col not in df_before.columns or col not in df_after.columns:
        continue

    before = df_before[col].dropna()
    after = df_after[col].dropna()

    # 전체 히스토그램 + tail 확대 + boxplot
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    # 전체 hist
    axes[0].hist(before, bins=40, alpha=0.6, label="Before")
    axes[0].hist(after, bins=40, alpha=0.6, label="After")
    axes[0].set_title(f"{col}\n전체 히스토그램")
    axes[0].legend()

    # tail 확대
    q95, q995 = tail_zoom_range(before, after)
    if q95 is not None:
        before_tail = before[before >= q95]
        after_tail = after[after >= q95]
        axes[1].hist(before_tail, bins=30, alpha=0.6, label="Before")
        axes[1].hist(after_tail, bins=30, alpha=0.6, label="After")
        axes[1].set_title(f"{col}\n상위 5% tail 확대")
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, "No tail data", ha="center", va="center")

    # boxplot
    axes[2].boxplot([before, after], labels=["Before", "After"])
    axes[2].set_title(f"{col}\nBoxplot")

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/d_type_compare/{col}_compare.png", dpi=150)
    plt.close()

# D 유형 요약표
d_summary = summary[summary["type"] == "D_normal_variation"].copy()
d_summary["changed_ratio_pct"] = (d_summary["changed_ratio"] * 100).round(2)
d_summary = d_summary.sort_values("changed_ratio", ascending=False)
d_summary.to_csv(f"{OUT_DIR}/d_type_compare/d_type_summary.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 6. B/C 유형: flag 효과 분석
#    - flag 비율
#    - flag별 불량률
#    - 원본 분포 + 기준선
# =========================================================
bc_rows = summary[summary["action"] == "flag_only"].copy()

flag_analysis_rows = []

for _, row in bc_rows.iterrows():
    col = row["column"]
    low = row["low"] if not pd.isna(row["low"]) else None
    high = row["high"] if not pd.isna(row["high"]) else None

    flag_col_candidates = [f"{col}_flag_spec", f"{col}_flag_process"]
    flag_col = None
    for cand in flag_col_candidates:
        if cand in df_after.columns:
            flag_col = cand
            break

    if flag_col is None or col not in df_before.columns:
        continue

    flag_rate = df_after[flag_col].mean()

    fail_rate_flag0 = np.nan
    fail_rate_flag1 = np.nan
    if "judge_bin" in df_after.columns:
        if (df_after[flag_col] == 0).sum() > 0:
            fail_rate_flag0 = df_after.loc[df_after[flag_col] == 0, "judge_bin"].mean()
        if (df_after[flag_col] == 1).sum() > 0:
            fail_rate_flag1 = df_after.loc[df_after[flag_col] == 1, "judge_bin"].mean()

    flag_analysis_rows.append({
        "column": col,
        "flag_col": flag_col,
        "type": row["type"],
        "flag_rate": flag_rate,
        "flag_rate_pct": round(flag_rate * 100, 2),
        "fail_rate_flag0": fail_rate_flag0,
        "fail_rate_flag1": fail_rate_flag1
    })

    fig, axes = plt.subplots(1, 2 if "judge_bin" in df_after.columns else 1, figsize=(12, 4))
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])

    # 원본 히스토그램 + 기준선
    axes[0].hist(df_before[col].dropna(), bins=40, alpha=0.8)
    if low is not None:
        axes[0].axvline(low, linestyle="--", linewidth=2, label=f"low={low}")
    if high is not None:
        axes[0].axvline(high, linestyle="--", linewidth=2, label=f"high={high}")
    axes[0].set_title(f"{col}\n원본 분포 + 기준선")
    axes[0].legend()

    # flag별 불량률
    if "judge_bin" in df_after.columns:
        vals = [fail_rate_flag0, fail_rate_flag1]
        labels = ["flag=0", "flag=1"]
        axes[1].bar(labels, vals)
        axes[1].set_title(f"{col}\nflag별 불량률")
        axes[1].set_ylabel("Fail rate")

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/bc_type_flags/{col}_flag_effect.png", dpi=150)
    plt.close()

flag_analysis = pd.DataFrame(flag_analysis_rows).sort_values("flag_rate", ascending=False)
flag_analysis.to_csv(f"{OUT_DIR}/bc_type_flags/flag_analysis.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 7. 최종 설명문 자동 생성
# =========================================================
comments = []

# A형
a_total_change = summary.loc[summary["type"] == "A_data_error", "changed_count"].sum()
if a_total_change == 0:
    comments.append("A형(데이터 오류형)은 하드리밋 위반 값이 없어 실제 변경이 발생하지 않았으므로 전후 히스토그램 차이가 나타나지 않음.")
else:
    comments.append(f"A형(데이터 오류형)은 총 {a_total_change}건이 제거/NaN 처리되어 일부 분포 정리 효과가 존재함.")

# B/C형
bc_count = len(summary[summary["action"] == "flag_only"])
comments.append(f"B/C형 변수 {bc_count}개는 값 자체를 바꾸지 않고 flag만 생성한 처리이므로 전후 히스토그램이 거의 동일하게 보이는 것이 정상임.")

# D형
d_top = d_summary.head(5)[["column", "changed_ratio_pct"]]
d_msg = "D형(IQR clip)에서는 tail 일부가 조정되었으며, 변경 비율 상위 변수는 " + ", ".join(
    [f"{r['column']}({r['changed_ratio_pct']}%)" for _, r in d_top.iterrows()]
) + " 임."
comments.append(d_msg)

# flag 극단 케이스
if not flag_analysis.empty:
    top_flag = flag_analysis.head(5)[["column", "flag_rate_pct"]]
    flag_msg = "flag 기반 탐지 효과가 큰 변수는 " + ", ".join(
        [f"{r['column']}({r['flag_rate_pct']}%)" for _, r in top_flag.iterrows()]
    ) + " 임."
    comments.append(flag_msg)

with open(f"{OUT_DIR}/final_interpretation.txt", "w", encoding="utf-8") as f:
    for i, c in enumerate(comments, 1):
        f.write(f"{i}. {c}\n")

# =========================================================
# 8. 핵심 체크용 화면 출력
# =========================================================
print("완료")
print(f"- 유형 요약표: {OUT_DIR}/type_summary/type_overview.csv")
print(f"- 유형 요약 그래프: {OUT_DIR}/type_summary/")
print(f"- D형 비교 그래프: {OUT_DIR}/d_type_compare/")
print(f"- B/C형 flag 분석: {OUT_DIR}/bc_type_flags/")
print(f"- 최종 해석문: {OUT_DIR}/final_interpretation.txt")

print("\n[핵심 해석 미리보기]")
for c in comments:
    print("-", c)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 0. 파일 경로
# =========================================================
INPUT_FILE = r"bat_process.csv"
OUT_DIR = r"outlier_redefined_results"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/plots_A_D", exist_ok=True)
os.makedirs(f"{OUT_DIR}/plots_B_C", exist_ok=True)

# =========================================================
# 1. 데이터 로드
# =========================================================
df = pd.read_csv(INPUT_FILE, encoding="euc-kr")
df_new = df.copy()

if "judge" in df_new.columns:
    df_new["judge_bin"] = (df_new["judge"] == "불량").astype(int)

n_rows = len(df_new)

# =========================================================
# 2. 변수 그룹 정의
# =========================================================
A_DATA_ERROR = [
    "ocv1_ocv", "ocv2_ocv", "socv1_ocv", "socv2_ocv", "socv3_ocv",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "pg1_imp", "pg1_impfit", "pc1_imp", "m1_res_ac",
    "m1_thick",
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
]

B_RECIPE = [
    "c1_curr_end", "dc1_curr_end", "c2_curr_end", "dc2_curr_end",
    "c3_curr_end", "dc3_curr_end", "c4_curr_end",
    "c3_cvval", "c4_cvval",
    "c3_ccval", "c4_ccval",
]

C_PROCESS = [
    "ocv2_deltaocv", "pg1_imp", "pc1_imp", "m1_res_ac", "m1_thick",
    "c3_time_cv", "c4_time_cv",
]

D_NORMAL = [
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "c1_voltage_avg", "dc1_voltage_avg",
    "c2_voltage_avg", "dc2_voltage_avg",
    "c3_voltage_avg", "dc3_voltage_avg",
    "c4_voltage_avg",
]

# =========================================================
# 3. 유틸 함수
# =========================================================
def apply_hard_limit(series, low=None, high=None):
    s = series.copy()
    if low is not None:
        s = s.mask(s < low, np.nan)
    if high is not None:
        s = s.mask(s > high, np.nan)
    return s

def iqr_bounds(series, k=1.5):
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    return low, high

def iqr_clip(series, k=1.5):
    low, high = iqr_bounds(series, k=k)
    return series.clip(lower=low, upper=high), low, high

def plot_before_after(before, after, col, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].hist(before.dropna(), bins=40, alpha=0.7, label="Before")
    axes[0].hist(after.dropna(), bins=40, alpha=0.7, label="After")
    axes[0].set_title(f"{col} - Histogram")
    axes[0].legend()

    # 상위 5% tail 확대
    all_s = pd.concat([before, after], axis=0).dropna()
    if len(all_s) > 0:
        q95 = all_s.quantile(0.95)
        axes[1].hist(before[before >= q95].dropna(), bins=30, alpha=0.7, label="Before")
        axes[1].hist(after[after >= q95].dropna(), bins=30, alpha=0.7, label="After")
        axes[1].set_title(f"{col} - Tail (Top 5%)")
        axes[1].legend()

    axes[2].boxplot([before.dropna(), after.dropna()], labels=["Before", "After"])
    axes[2].set_title(f"{col} - Boxplot")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

def plot_with_threshold(series, low, high, col, save_path):
    plt.figure(figsize=(6, 4))
    plt.hist(series.dropna(), bins=40, alpha=0.8)
    if low is not None:
        plt.axvline(low, linestyle="--", linewidth=2, label=f"low={low:.3f}")
    if high is not None:
        plt.axvline(high, linestyle="--", linewidth=2, label=f"high={high:.3f}")
    plt.title(f"{col} - Distribution with Threshold")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

summary_rows = []

# =========================================================
# 4. A형 재정의: sanity check만 수행
# =========================================================
A_LIMITS = {}

for col in A_DATA_ERROR:
    if col not in df_new.columns:
        continue

    # 변수명 기준으로 최소 sanity 정의
    if "temp" in col:
        A_LIMITS[col] = (0, 1000)
    elif "time" in col:
        A_LIMITS[col] = (0, 100000)
    elif "ocv" in col or "voltage" in col:
        A_LIMITS[col] = (0, 5000)
    elif "imp" in col or "res" in col:
        A_LIMITS[col] = (0, 10000)
    elif "thick" in col:
        A_LIMITS[col] = (0, 10000)
    else:
        A_LIMITS[col] = (None, None)

for col in A_DATA_ERROR:
    if col not in df_new.columns:
        continue

    before = df_new[col].copy()
    low, high = A_LIMITS[col]
    after = apply_hard_limit(before, low=low, high=high)
    df_new[col] = after

    changed = before.notna().sum() - after.notna().sum()

    summary_rows.append({
        "column": col,
        "type": "A_data_error",
        "action": "hard_limit_to_nan",
        "low": low,
        "high": high,
        "changed_count": int(changed),
        "changed_ratio": changed / n_rows
    })

    plot_before_after(before, after, col, f"{OUT_DIR}/plots_A_D/{col}_A_compare.png")

# =========================================================
# 5. B형 재정의: 중앙값 ± 허용오차
#    근거: 설정값/종단값은 분포형이 아니라 setpoint deviation이 핵심
# =========================================================
for col in B_RECIPE:
    if col not in df_new.columns:
        continue

    s = df_new[col].dropna()
    if len(s) == 0:
        continue

    med = s.median()

    # 허용오차 규칙
    # curr_end, ccval처럼 값 범위가 작을 수 있는 변수는 ±2%
    # cvval처럼 setpoint 전압이면 ±1%
    if "cvval" in col:
        tol = abs(med) * 0.01
    else:
        tol = abs(med) * 0.02

    low = med - tol
    high = med + tol

    flag_col = f"{col}_flag_recipe"
    df_new[flag_col] = ((df_new[col] < low) | (df_new[col] > high)).astype(int)

    changed = df_new[flag_col].sum()

    summary_rows.append({
        "column": col,
        "type": "B_recipe_deviation",
        "action": "flag_only_median_tol",
        "low": low,
        "high": high,
        "changed_count": int(changed),
        "changed_ratio": changed / n_rows
    })

    plot_with_threshold(df_new[col], low, high, col, f"{OUT_DIR}/plots_B_C/{col}_B_threshold.png")

# =========================================================
# 6. C형 재정의: 상위 tail 기준
#    근거: 공정 이상형은 절대 삭제보다 상위 극단값을 위험군으로 보는 것이 타당
# =========================================================
for col in C_PROCESS:
    if col not in df_new.columns:
        continue

    s = df_new[col].dropna()
    if len(s) == 0:
        continue

    p95 = s.quantile(0.95)
    p99 = s.quantile(0.99)

    df_new[f"{col}_flag_p95"] = (df_new[col] > p95).astype(int)
    df_new[f"{col}_flag_p99"] = (df_new[col] > p99).astype(int)

    changed = df_new[f"{col}_flag_p99"].sum()

    summary_rows.append({
        "column": col,
        "type": "C_process_anomaly",
        "action": "flag_only_p95_p99",
        "low": p95,
        "high": p99,
        "changed_count": int(changed),
        "changed_ratio": changed / n_rows
    })

    plot_with_threshold(df_new[col], p95, p99, col, f"{OUT_DIR}/plots_B_C/{col}_C_threshold.png")

# =========================================================
# 7. D형 재정의: IQR clip + flag 병행
#    근거: 자연 변동형은 극단 꼬리를 다듬되 정보는 flag로 남기는 것이 좋음
# =========================================================
for col in D_NORMAL:
    if col not in df_new.columns:
        continue

    before = df_new[col].copy()
    clipped, low, high = iqr_clip(before, k=1.5)

    df_new[f"{col}_clip"] = clipped
    df_new[f"{col}_flag_iqr"] = ((before < low) | (before > high)).astype(int)

    changed = df_new[f"{col}_flag_iqr"].sum()

    summary_rows.append({
        "column": col,
        "type": "D_normal_variation",
        "action": "iqr_clip_and_flag",
        "low": low,
        "high": high,
        "changed_count": int(changed),
        "changed_ratio": changed / n_rows
    })

    plot_before_after(before, clipped, col, f"{OUT_DIR}/plots_A_D/{col}_D_compare.png")

# =========================================================
# 8. 결과 저장
# =========================================================
summary_new = pd.DataFrame(summary_rows).sort_values(["type", "changed_ratio"], ascending=[True, False])
summary_new.to_csv(f"{OUT_DIR}/outlier_redefined_summary.csv", index=False, encoding="utf-8-sig")
df_new.to_csv(f"{OUT_DIR}/bat_process_redefined_outlier_handling.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 9. 유형별 요약표
# =========================================================
type_summary = (
    summary_new.groupby(["type", "action"], as_index=False)
    .agg(
        variable_count=("column", "count"),
        total_changed=("changed_count", "sum"),
        mean_changed_ratio=("changed_ratio", "mean"),
        max_changed_ratio=("changed_ratio", "max")
    )
)
type_summary["mean_changed_ratio_pct"] = (type_summary["mean_changed_ratio"] * 100).round(2)
type_summary["max_changed_ratio_pct"] = (type_summary["max_changed_ratio"] * 100).round(2)
type_summary.to_csv(f"{OUT_DIR}/type_summary.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 10. flag 비율 요약
# =========================================================
flag_cols = [c for c in df_new.columns if c.endswith("_flag_recipe") or c.endswith("_flag_p95") or c.endswith("_flag_p99") or c.endswith("_flag_iqr")]
flag_summary = pd.DataFrame({
    "flag_col": flag_cols,
    "flag_count": [df_new[c].sum() for c in flag_cols],
    "flag_ratio": [df_new[c].mean() for c in flag_cols]
}).sort_values("flag_ratio", ascending=False)
flag_summary["flag_ratio_pct"] = (flag_summary["flag_ratio"] * 100).round(2)
flag_summary.to_csv(f"{OUT_DIR}/flag_summary.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 11. 해석 문장 자동 생성
# =========================================================
comments = []

a_total = summary_new.loc[summary_new["type"] == "A_data_error", "changed_count"].sum()
comments.append(
    f"A형은 sanity check 목적의 하드리밋만 적용했으며, 실제 변경 건수는 {a_total}건이다. "
    f"이는 데이터 파손 여부 점검용으로 해석해야 한다."
)

b_top = summary_new[summary_new["type"] == "B_recipe_deviation"].sort_values("changed_ratio", ascending=False).head(5)
comments.append(
    "B형은 중앙값 ± 허용오차 기준으로 재정의했다. "
    "이는 설정값형 변수의 특성상 분포형 이상치보다 setpoint 이탈 탐지가 더 타당하기 때문이다. "
    "상위 이탈 변수: " +
    ", ".join([f"{r['column']}({r['changed_ratio']*100:.2f}%)" for _, r in b_top.iterrows()])
)

c_top = summary_new[summary_new["type"] == "C_process_anomaly"].sort_values("changed_ratio", ascending=False).head(5)
comments.append(
    "C형은 절대값 기준 대신 상위 95%/99% tail 기준으로 재정의했다. "
    "이는 공정 이상형 변수가 불량 신호를 담고 있어 삭제보다 상대적 극단값 탐지가 더 적절하기 때문이다. "
    "상위 P99 위험 변수: " +
    ", ".join([f"{r['column']}({r['changed_ratio']*100:.2f}%)" for _, r in c_top.iterrows()])
)

d_top = summary_new[summary_new["type"] == "D_normal_variation"].sort_values("changed_ratio", ascending=False).head(5)
comments.append(
    "D형은 IQR clip과 flag를 동시에 생성했다. "
    "이는 자연 변동형 변수의 극단 꼬리를 모델링용으로 안정화하면서도, 이상 신호 자체는 보존하기 위한 방식이다. "
    "상위 IQR 변수: " +
    ", ".join([f"{r['column']}({r['changed_ratio']*100:.2f}%)" for _, r in d_top.iterrows()])
)

with open(f"{OUT_DIR}/final_interpretation.txt", "w", encoding="utf-8") as f:
    for i, c in enumerate(comments, 1):
        f.write(f"{i}. {c}\n")

print("완료")
print(f"- 요약표: {OUT_DIR}/outlier_redefined_summary.csv")
print(f"- 유형요약: {OUT_DIR}/type_summary.csv")
print(f"- flag요약: {OUT_DIR}/flag_summary.csv")
print(f"- 처리 후 데이터: {OUT_DIR}/bat_process_redefined_outlier_handling.csv")
print(f"- 해석문: {OUT_DIR}/final_interpretation.txt")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 0. 파일 경로
# =========================================================
INPUT_FILE = r"bat_process.csv"
OUT_DIR = r"outlier_redefined_v2_results"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/plots_A_D", exist_ok=True)
os.makedirs(f"{OUT_DIR}/plots_B_C", exist_ok=True)

# =========================================================
# 1. 데이터 로드
# =========================================================
df = pd.read_csv(INPUT_FILE, encoding="euc-kr")
df_new = df.copy()

if "judge" in df_new.columns:
    df_new["judge_bin"] = (df_new["judge"] == "불량").astype(int)

n_rows = len(df_new)

# =========================================================
# 2. 변수 그룹 정의
# =========================================================
A_DATA_ERROR = [
    "ocv1_ocv", "ocv2_ocv", "socv1_ocv", "socv2_ocv", "socv3_ocv",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "pg1_imp", "pg1_impfit", "pc1_imp", "m1_res_ac",
    "m1_thick",
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
]

B_RECIPE = [
    "c1_curr_end", "dc1_curr_end", "c2_curr_end", "dc2_curr_end",
    "c3_curr_end", "dc3_curr_end", "c4_curr_end",
    "c3_cvval", "c4_cvval",
    "c3_ccval", "c4_ccval",
]

C_PROCESS = [
    "ocv2_deltaocv", "pg1_imp", "pc1_imp", "m1_res_ac", "m1_thick",
    "c3_time_cv", "c4_time_cv",
]

D_NORMAL = [
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "c1_voltage_avg", "dc1_voltage_avg",
    "c2_voltage_avg", "dc2_voltage_avg",
    "c3_voltage_avg", "dc3_voltage_avg",
    "c4_voltage_avg",
]

# =========================================================
# 3. 유틸 함수
# =========================================================
def apply_hard_limit(series, low=None, high=None):
    s = series.copy()
    if low is not None:
        s = s.mask(s < low, np.nan)
    if high is not None:
        s = s.mask(s > high, np.nan)
    return s

def iqr_bounds(series, k=1.5):
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    return low, high

def iqr_clip(series, k=1.5):
    low, high = iqr_bounds(series, k=k)
    return series.clip(lower=low, upper=high), low, high

def plot_before_after(before, after, col, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].hist(before.dropna(), bins=40, alpha=0.7, label="Before")
    axes[0].hist(after.dropna(), bins=40, alpha=0.7, label="After")
    axes[0].set_title(f"{col} - Histogram")
    axes[0].legend()

    all_s = pd.concat([before, after], axis=0).dropna()
    if len(all_s) > 0:
        q95 = all_s.quantile(0.95)
        axes[1].hist(before[before >= q95].dropna(), bins=30, alpha=0.7, label="Before")
        axes[1].hist(after[after >= q95].dropna(), bins=30, alpha=0.7, label="After")
        axes[1].set_title(f"{col} - Tail (Top 5%)")
        axes[1].legend()

    axes[2].boxplot([before.dropna(), after.dropna()], labels=["Before", "After"])
    axes[2].set_title(f"{col} - Boxplot")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

def plot_with_thresholds(series, thresholds, col, save_path):
    plt.figure(figsize=(7, 4))
    plt.hist(series.dropna(), bins=40, alpha=0.8)
    for t in thresholds:
        plt.axvline(t, linestyle="--", linewidth=1.5)
    plt.title(f"{col} - Distribution with Recipe Centers")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

def get_main_recipe_centers(series, top_k=3, min_ratio=0.05, round_digits=None):
    """
    레시피형 변수의 주요 setpoint 후보 추출
    - 값의 빈도가 높은 순으로 top_k개 추출
    - 전체의 min_ratio 이상 비중을 가지는 값만 채택
    - 연속형인데 미세 노이즈가 있으면 round_digits로 반올림 후 중심값 추출 가능
    """
    s = series.dropna().copy()
    if len(s) == 0:
        return []

    if round_digits is not None:
        s = s.round(round_digits)

    vc = s.value_counts(normalize=True)
    centers = vc[vc >= min_ratio].index.tolist()[:top_k]

    # min_ratio 이상 값이 없으면 top 1만이라도 사용
    if len(centers) == 0:
        centers = vc.index.tolist()[:1]

    return sorted(centers)

def make_recipe_flag(series, centers, tol_abs=None, tol_ratio=None):
    """
    centers 중 하나라도 가까우면 정상
    아니면 flag=1
    """
    s = series.copy()

    if len(centers) == 0:
        return pd.Series(0, index=s.index, dtype=int)

    ok_mask = pd.Series(False, index=s.index)

    for c in centers:
        if tol_abs is not None:
            tol = tol_abs
        else:
            tol = abs(c) * tol_ratio

        ok_mask = ok_mask | ((s >= c - tol) & (s <= c + tol))

    return (~ok_mask).astype(int)

summary_rows = []

# =========================================================
# 4. A형: sanity check
# =========================================================
A_LIMITS = {}

for col in A_DATA_ERROR:
    if col not in df_new.columns:
        continue

    if "temp" in col:
        A_LIMITS[col] = (0, 1000)
    elif "time" in col:
        A_LIMITS[col] = (0, 100000)
    elif "ocv" in col or "voltage" in col:
        A_LIMITS[col] = (0, 5000)
    elif "imp" in col or "res" in col:
        A_LIMITS[col] = (0, 10000)
    elif "thick" in col:
        A_LIMITS[col] = (0, 10000)
    else:
        A_LIMITS[col] = (None, None)

for col in A_DATA_ERROR:
    if col not in df_new.columns:
        continue

    before = df_new[col].copy()
    low, high = A_LIMITS[col]
    after = apply_hard_limit(before, low=low, high=high)
    df_new[col] = after

    changed = before.notna().sum() - after.notna().sum()

    summary_rows.append({
        "column": col,
        "type": "A_data_error",
        "action": "hard_limit_to_nan",
        "low": low,
        "high": high,
        "changed_count": int(changed),
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

    plot_before_after(before, after, col, f"{OUT_DIR}/plots_A_D/{col}_A_compare.png")

# =========================================================
# 5. B형: mode 기반 + 다중 레시피 허용
# =========================================================
for col in B_RECIPE:
    if col not in df_new.columns:
        continue

    s = df_new[col].dropna()
    if len(s) == 0:
        continue

    # 변수 특성별 tolerance/반올림 규칙
    if "cvval" in col:
        round_digits = 0
        centers = get_main_recipe_centers(s, top_k=3, min_ratio=0.03, round_digits=round_digits)
        flag = make_recipe_flag(series=df_new[col], centers=centers, tol_abs=20, tol_ratio=None)
        # cv setpoint는 절대 오차 중심이 더 직관적
        low_display = min(centers) - 20 if len(centers) else np.nan
        high_display = max(centers) + 20 if len(centers) else np.nan

    elif "curr_end" in col:
        round_digits = 0
        centers = get_main_recipe_centers(s, top_k=3, min_ratio=0.03, round_digits=round_digits)
        flag = make_recipe_flag(series=df_new[col], centers=centers, tol_abs=None, tol_ratio=0.03)
        low_display = min(centers) * 0.97 if len(centers) else np.nan
        high_display = max(centers) * 1.03 if len(centers) else np.nan

    elif "ccval" in col:
        round_digits = 0
        centers = get_main_recipe_centers(s, top_k=3, min_ratio=0.03, round_digits=round_digits)
        flag = make_recipe_flag(series=df_new[col], centers=centers, tol_abs=None, tol_ratio=0.03)
        low_display = min(centers) * 0.97 if len(centers) else np.nan
        high_display = max(centers) * 1.03 if len(centers) else np.nan

    else:
        centers = get_main_recipe_centers(s, top_k=3, min_ratio=0.03, round_digits=0)
        flag = make_recipe_flag(series=df_new[col], centers=centers, tol_abs=None, tol_ratio=0.03)
        low_display = np.nan
        high_display = np.nan

    flag_col = f"{col}_flag_recipe"
    df_new[flag_col] = flag
    changed = int(df_new[flag_col].sum())

    summary_rows.append({
        "column": col,
        "type": "B_recipe_deviation",
        "action": "flag_only_mode_centers",
        "low": low_display,
        "high": high_display,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ",".join(map(str, centers))
    })

    plot_with_thresholds(df_new[col], centers, col, f"{OUT_DIR}/plots_B_C/{col}_B_centers.png")

# =========================================================
# 6. C형: 공정 이상형 = p95 / p99 tail
# =========================================================
for col in C_PROCESS:
    if col not in df_new.columns:
        continue

    s = df_new[col].dropna()
    if len(s) == 0:
        continue

    p95 = s.quantile(0.95)
    p99 = s.quantile(0.99)

    df_new[f"{col}_flag_p95"] = (df_new[col] > p95).astype(int)
    df_new[f"{col}_flag_p99"] = (df_new[col] > p99).astype(int)

    changed = int(df_new[f"{col}_flag_p99"].sum())

    summary_rows.append({
        "column": col,
        "type": "C_process_anomaly",
        "action": "flag_only_p95_p99",
        "low": p95,
        "high": p99,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

    plot_with_thresholds(df_new[col], [p95, p99], col, f"{OUT_DIR}/plots_B_C/{col}_C_threshold.png")

# =========================================================
# 7. D형: IQR clip + flag
# =========================================================
for col in D_NORMAL:
    if col not in df_new.columns:
        continue

    before = df_new[col].copy()
    clipped, low, high = iqr_clip(before, k=1.5)

    df_new[f"{col}_clip"] = clipped
    df_new[f"{col}_flag_iqr"] = ((before < low) | (before > high)).astype(int)

    changed = int(df_new[f"{col}_flag_iqr"].sum())

    summary_rows.append({
        "column": col,
        "type": "D_normal_variation",
        "action": "iqr_clip_and_flag",
        "low": low,
        "high": high,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

    plot_before_after(before, clipped, col, f"{OUT_DIR}/plots_A_D/{col}_D_compare.png")

# =========================================================
# 8. 결과 저장
# =========================================================
summary_new = pd.DataFrame(summary_rows).sort_values(["type", "changed_ratio"], ascending=[True, False])
summary_new.to_csv(f"{OUT_DIR}/outlier_redefined_v2_summary.csv", index=False, encoding="utf-8-sig")
df_new.to_csv(f"{OUT_DIR}/bat_process_redefined_v2_outlier_handling.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 9. 유형 요약
# =========================================================
type_summary = (
    summary_new.groupby(["type", "action"], as_index=False)
    .agg(
        variable_count=("column", "count"),
        total_changed=("changed_count", "sum"),
        mean_changed_ratio=("changed_ratio", "mean"),
        max_changed_ratio=("changed_ratio", "max")
    )
)
type_summary["mean_changed_ratio_pct"] = (type_summary["mean_changed_ratio"] * 100).round(2)
type_summary["max_changed_ratio_pct"] = (type_summary["max_changed_ratio"] * 100).round(2)
type_summary.to_csv(f"{OUT_DIR}/type_summary.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 10. flag 요약
# =========================================================
flag_cols = [
    c for c in df_new.columns
    if c.endswith("_flag_recipe") or c.endswith("_flag_p95") or c.endswith("_flag_p99") or c.endswith("_flag_iqr")
]
flag_summary = pd.DataFrame({
    "flag_col": flag_cols,
    "flag_count": [df_new[c].sum() for c in flag_cols],
    "flag_ratio": [df_new[c].mean() for c in flag_cols]
}).sort_values("flag_ratio", ascending=False)
flag_summary["flag_ratio_pct"] = (flag_summary["flag_ratio"] * 100).round(2)
flag_summary.to_csv(f"{OUT_DIR}/flag_summary.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 11. 자동 해석문
# =========================================================
comments = []

a_total = summary_new.loc[summary_new["type"] == "A_data_error", "changed_count"].sum()
comments.append(
    f"A형은 sanity check 목적이며 실제 변경 건수는 {a_total}건이다. "
    f"데이터 파손 여부 점검용으로 해석한다."
)

b_top = summary_new[summary_new["type"] == "B_recipe_deviation"].sort_values("changed_ratio", ascending=False).head(5)
comments.append(
    "B형은 중앙값 기준 대신 주요 setpoint(mode-like centers) 기반으로 재정의했다. "
    "이는 레시피형 변수에서 평균/중앙값보다 실제 설정값 군집이 더 중요하기 때문이다. "
    "상위 이탈 변수: " +
    ", ".join([f"{r['column']}({r['changed_ratio']*100:.2f}%)" for _, r in b_top.iterrows()])
)

c_top = summary_new[summary_new["type"] == "C_process_anomaly"].sort_values("changed_ratio", ascending=False).head(5)
comments.append(
    "C형은 상위 95%/99% tail 기준으로 유지했다. "
    "이 변수들은 불량 신호를 담고 있으므로 삭제보다 상대적 극단값 탐지가 더 적절하다. "
    "상위 P99 위험 변수: " +
    ", ".join([f"{r['column']}({r['changed_ratio']*100:.2f}%)" for _, r in c_top.iterrows()])
)

d_top = summary_new[summary_new["type"] == "D_normal_variation"].sort_values("changed_ratio", ascending=False).head(5)
comments.append(
    "D형은 IQR clip과 flag를 동시에 생성했다. "
    "이는 자연 변동형 변수의 극단 꼬리를 모델링용으로 안정화하면서도 이상 신호는 보존하기 위한 방식이다. "
    "상위 IQR 변수: " +
    ", ".join([f"{r['column']}({r['changed_ratio']*100:.2f}%)" for _, r in d_top.iterrows()])
)

with open(f"{OUT_DIR}/final_interpretation.txt", "w", encoding="utf-8") as f:
    for i, c in enumerate(comments, 1):
        f.write(f"{i}. {c}\n")

print("완료")
print(f"- 요약표: {OUT_DIR}/outlier_redefined_v2_summary.csv")
print(f"- 유형요약: {OUT_DIR}/type_summary.csv")
print(f"- flag요약: {OUT_DIR}/flag_summary.csv")
print(f"- 처리 후 데이터: {OUT_DIR}/bat_process_redefined_v2_outlier_handling.csv")
print(f"- 해석문: {OUT_DIR}/final_interpretation.txt")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 0. 파일 경로
# =========================================================
INPUT_FILE = "bat_process.csv"
OUT_DIR = "outlier_final_results"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/plots_A_D", exist_ok=True)
os.makedirs(f"{OUT_DIR}/plots_B_C", exist_ok=True)

# =========================================================
# 1. 데이터 로드
# =========================================================
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"입력 파일이 없습니다: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE, encoding="euc-kr")
df_new = df.copy()

if "judge" in df_new.columns:
    df_new["judge_bin"] = (df_new["judge"] == "불량").astype(int)

n_rows = len(df_new)

# =========================================================
# 2. 변수 그룹 정의
# =========================================================
A_DATA_ERROR = [
    "ocv1_ocv", "ocv2_ocv", "socv1_ocv", "socv2_ocv", "socv3_ocv",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "pg1_imp", "pg1_impfit", "pc1_imp", "m1_res_ac",
    "m1_thick",
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
]

B_RECIPE = [
    "c1_curr_end", "dc1_curr_end", "c2_curr_end", "dc2_curr_end",
    "c3_curr_end", "dc3_curr_end", "c4_curr_end",
    "c3_cvval", "c4_cvval",
    "c3_ccval", "c4_ccval",
]

C_PROCESS = [
    "ocv2_deltaocv", "pg1_imp", "pc1_imp", "m1_res_ac", "m1_thick",
    "c3_time_cv", "c4_time_cv",
]

D_NORMAL = [
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "c1_voltage_avg", "dc1_voltage_avg",
    "c2_voltage_avg", "dc2_voltage_avg",
    "c3_voltage_avg", "dc3_voltage_avg",
    "c4_voltage_avg",
]

# =========================================================
# 3. B형 변수별 설정
#    - round_digits: 군집화 안정화
#    - top_k: 허용할 주요 setpoint 개수
#    - min_ratio: 최소 점유율
#    - tol_abs / tol_ratio: 허용 오차
# =========================================================
B_CONFIG = {
    # cvval은 절대 오차가 더 직관적
    "c3_cvval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": 20, "tol_ratio": None},
    "c4_cvval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": 20, "tol_ratio": None},

    # curr_end는 좀 더 촘촘하게 볼 수 있게 2% 허용
    "c1_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "dc1_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "c2_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "dc2_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "c3_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "dc3_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "c4_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},

    # ccval은 3% 유지
    "c3_ccval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.03},
    "c4_ccval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.03},
}

# =========================================================
# 4. 유틸 함수
# =========================================================
def apply_hard_limit(series, low=None, high=None):
    s = series.copy()
    if low is not None:
        s = s.mask(s < low, np.nan)
    if high is not None:
        s = s.mask(s > high, np.nan)
    return s

def iqr_bounds(series, k=1.5):
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    return low, high

def iqr_clip(series, k=1.5):
    low, high = iqr_bounds(series, k=k)
    return series.clip(lower=low, upper=high), low, high

def plot_before_after(before, after, col, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].hist(before.dropna(), bins=40, alpha=0.7, label="Before")
    axes[0].hist(after.dropna(), bins=40, alpha=0.7, label="After")
    axes[0].set_title(f"{col} - Histogram")
    axes[0].legend()

    all_s = pd.concat([before, after], axis=0).dropna()
    if len(all_s) > 0:
        q95 = all_s.quantile(0.95)
        axes[1].hist(before[before >= q95].dropna(), bins=30, alpha=0.7, label="Before")
        axes[1].hist(after[after >= q95].dropna(), bins=30, alpha=0.7, label="After")
        axes[1].set_title(f"{col} - Tail (Top 5%)")
        axes[1].legend()

    axes[2].boxplot([before.dropna(), after.dropna()], labels=["Before", "After"])
    axes[2].set_title(f"{col} - Boxplot")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

def plot_with_thresholds(series, thresholds, col, save_path):
    plt.figure(figsize=(7, 4))
    plt.hist(series.dropna(), bins=40, alpha=0.8)
    for t in thresholds:
        plt.axvline(t, linestyle="--", linewidth=1.5)
    plt.title(f"{col} - Distribution with Centers/Thresholds")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

def get_main_recipe_centers(series, top_k=3, min_ratio=0.05, round_digits=None):
    s = series.dropna().copy()
    if len(s) == 0:
        return []

    if round_digits is not None:
        s = s.round(round_digits)

    vc = s.value_counts(normalize=True)
    centers = vc[vc >= min_ratio].index.tolist()[:top_k]

    if len(centers) == 0:
        centers = vc.index.tolist()[:1]

    return sorted(centers)

def make_recipe_flag(series, centers, tol_abs=None, tol_ratio=None):
    s = series.copy()

    if len(centers) == 0:
        return pd.Series(0, index=s.index, dtype=int)

    ok_mask = pd.Series(False, index=s.index)

    for c in centers:
        if tol_abs is not None:
            tol = tol_abs
        else:
            tol = abs(c) * tol_ratio
        ok_mask = ok_mask | ((s >= c - tol) & (s <= c + tol))

    return (~ok_mask).astype(int)

summary_rows = []

# =========================================================
# 5. A형: sanity check
# =========================================================
A_LIMITS = {}
for col in A_DATA_ERROR:
    if col not in df_new.columns:
        continue
    if "temp" in col:
        A_LIMITS[col] = (0, 1000)
    elif "time" in col:
        A_LIMITS[col] = (0, 100000)
    elif "ocv" in col or "voltage" in col:
        A_LIMITS[col] = (0, 5000)
    elif "imp" in col or "res" in col:
        A_LIMITS[col] = (0, 10000)
    elif "thick" in col:
        A_LIMITS[col] = (0, 10000)
    else:
        A_LIMITS[col] = (None, None)

for col in A_DATA_ERROR:
    if col not in df_new.columns:
        continue

    before = df_new[col].copy()
    low, high = A_LIMITS[col]
    after = apply_hard_limit(before, low=low, high=high)
    df_new[col] = after

    changed = before.notna().sum() - after.notna().sum()

    summary_rows.append({
        "column": col,
        "type": "A_data_error",
        "action": "hard_limit_to_nan",
        "low": low,
        "high": high,
        "changed_count": int(changed),
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

    plot_before_after(before, after, col, f"{OUT_DIR}/plots_A_D/{col}_A_compare.png")

# =========================================================
# 6. B형: mode-like centers 기반
# =========================================================
for col in B_RECIPE:
    if col not in df_new.columns:
        continue

    s = df_new[col].dropna()
    if len(s) == 0:
        continue

    cfg = B_CONFIG.get(col, {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.03})

    centers = get_main_recipe_centers(
        s,
        top_k=cfg["top_k"],
        min_ratio=cfg["min_ratio"],
        round_digits=cfg["round_digits"]
    )

    flag = make_recipe_flag(
        series=df_new[col],
        centers=centers,
        tol_abs=cfg["tol_abs"],
        tol_ratio=cfg["tol_ratio"]
    )

    flag_col = f"{col}_flag_recipe"
    df_new[flag_col] = flag
    changed = int(df_new[flag_col].sum())

    if cfg["tol_abs"] is not None:
        low_display = min(centers) - cfg["tol_abs"] if len(centers) else np.nan
        high_display = max(centers) + cfg["tol_abs"] if len(centers) else np.nan
    else:
        low_display = min(centers) * (1 - cfg["tol_ratio"]) if len(centers) else np.nan
        high_display = max(centers) * (1 + cfg["tol_ratio"]) if len(centers) else np.nan

    summary_rows.append({
        "column": col,
        "type": "B_recipe_deviation",
        "action": "flag_only_mode_centers",
        "low": low_display,
        "high": high_display,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ",".join(map(str, centers))
    })

    plot_with_thresholds(df_new[col], centers, col, f"{OUT_DIR}/plots_B_C/{col}_B_centers.png")

# =========================================================
# 7. C형: 공정 이상형 = p95/p99
# =========================================================
for col in C_PROCESS:
    if col not in df_new.columns:
        continue

    s = df_new[col].dropna()
    if len(s) == 0:
        continue

    p95 = s.quantile(0.95)
    p99 = s.quantile(0.99)

    df_new[f"{col}_flag_p95"] = (df_new[col] > p95).astype(int)
    df_new[f"{col}_flag_p99"] = (df_new[col] > p99).astype(int)

    changed = int(df_new[f"{col}_flag_p99"].sum())

    summary_rows.append({
        "column": col,
        "type": "C_process_anomaly",
        "action": "flag_only_p95_p99",
        "low": p95,
        "high": p99,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

    plot_with_thresholds(df_new[col], [p95, p99], col, f"{OUT_DIR}/plots_B_C/{col}_C_threshold.png")

# =========================================================
# 8. D형: IQR clip + flag
# =========================================================
for col in D_NORMAL:
    if col not in df_new.columns:
        continue

    before = df_new[col].copy()
    clipped, low, high = iqr_clip(before, k=1.5)

    df_new[f"{col}_clip"] = clipped
    df_new[f"{col}_flag_iqr"] = ((before < low) | (before > high)).astype(int)

    changed = int(df_new[f"{col}_flag_iqr"].sum())

    summary_rows.append({
        "column": col,
        "type": "D_normal_variation",
        "action": "iqr_clip_and_flag",
        "low": low,
        "high": high,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

    plot_before_after(before, clipped, col, f"{OUT_DIR}/plots_A_D/{col}_D_compare.png")

# =========================================================
# 9. 결과 저장
# =========================================================
summary_new = pd.DataFrame(summary_rows).sort_values(["type", "changed_ratio"], ascending=[True, False])
summary_new.to_csv(f"{OUT_DIR}/outlier_final_summary.csv", index=False, encoding="utf-8-sig")
df_new.to_csv(f"{OUT_DIR}/bat_process_outlier_final.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 10. 유형 요약
# =========================================================
type_summary = (
    summary_new.groupby(["type", "action"], as_index=False)
    .agg(
        variable_count=("column", "count"),
        total_changed=("changed_count", "sum"),
        mean_changed_ratio=("changed_ratio", "mean"),
        max_changed_ratio=("changed_ratio", "max")
    )
)
type_summary["mean_changed_ratio_pct"] = (type_summary["mean_changed_ratio"] * 100).round(2)
type_summary["max_changed_ratio_pct"] = (type_summary["max_changed_ratio"] * 100).round(2)
type_summary.to_csv(f"{OUT_DIR}/type_summary.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 11. flag 요약
# =========================================================
flag_cols = [
    c for c in df_new.columns
    if c.endswith("_flag_recipe") or c.endswith("_flag_p95") or c.endswith("_flag_p99") or c.endswith("_flag_iqr")
]
flag_summary = pd.DataFrame({
    "flag_col": flag_cols,
    "flag_count": [df_new[c].sum() for c in flag_cols],
    "flag_ratio": [df_new[c].mean() for c in flag_cols]
}).sort_values("flag_ratio", ascending=False)
flag_summary["flag_ratio_pct"] = (flag_summary["flag_ratio"] * 100).round(2)
flag_summary.to_csv(f"{OUT_DIR}/flag_summary.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 12. 자동 해석문
# =========================================================
comments = []

a_total = summary_new.loc[summary_new["type"] == "A_data_error", "changed_count"].sum()
comments.append(
    f"A형은 sanity check 목적이며 실제 변경 건수는 {a_total}건이다. 데이터 파손 여부 점검용으로 해석한다."
)

b_top = summary_new[summary_new["type"] == "B_recipe_deviation"].sort_values("changed_ratio", ascending=False).head(5)
comments.append(
    "B형은 주요 setpoint(mode-like centers) 기반으로 재정의했다. "
    "이는 레시피형 변수에서 평균/중앙값보다 실제 설정값 군집이 더 중요하기 때문이다. "
    "상위 이탈 변수: " +
    ", ".join([f"{r['column']}({r['changed_ratio']*100:.2f}%)" for _, r in b_top.iterrows()])
)

c_top = summary_new[summary_new["type"] == "C_process_anomaly"].sort_values("changed_ratio", ascending=False).head(5)
comments.append(
    "C형은 상위 95%/99% tail 기준으로 유지했다. "
    "이 변수들은 불량 신호를 담고 있으므로 삭제보다 상대적 극단값 탐지가 더 적절하다. "
    "상위 P99 위험 변수: " +
    ", ".join([f"{r['column']}({r['changed_ratio']*100:.2f}%)" for _, r in c_top.iterrows()])
)

d_top = summary_new[summary_new["type"] == "D_normal_variation"].sort_values("changed_ratio", ascending=False).head(5)
comments.append(
    "D형은 IQR clip과 flag를 동시에 생성했다. "
    "이는 자연 변동형 변수의 극단 꼬리를 모델링용으로 안정화하면서도 이상 신호는 보존하기 위한 방식이다. "
    "상위 IQR 변수: " +
    ", ".join([f"{r['column']}({r['changed_ratio']*100:.2f}%)" for _, r in d_top.iterrows()])
)

with open(f"{OUT_DIR}/final_interpretation.txt", "w", encoding="utf-8") as f:
    for i, c in enumerate(comments, 1):
        f.write(f"{i}. {c}\n")

print("완료")
print(f"- 요약표: {OUT_DIR}/outlier_final_summary.csv")
print(f"- 유형요약: {OUT_DIR}/type_summary.csv")
print(f"- flag요약: {OUT_DIR}/flag_summary.csv")
print(f"- 처리 후 데이터: {OUT_DIR}/bat_process_outlier_final.csv")
print(f"- 해석문: {OUT_DIR}/final_interpretation.txt")

In [ ]:
import os
import glob
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 0. 공통 설정
# =========================================================
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_DIR = f"visual_check_run_{timestamp}"
OUT_DIR = os.path.join(BASE_DIR, "outputs")
LOG_DIR = os.path.join(BASE_DIR, "logs")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# =========================================================
# 1. 파일 자동 탐색 함수
#    - 동일 파일이 여러 개면 가장 최근 수정 파일 사용
# =========================================================
def find_latest_file(filename_pattern: str) -> str:
    matches = glob.glob(f"**/{filename_pattern}", recursive=True)
    matches = [m for m in matches if os.path.isfile(m)]

    if len(matches) == 0:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {filename_pattern}")

    # 가장 최근 수정된 파일 선택
    matches = sorted(matches, key=lambda x: os.path.getmtime(x), reverse=True)
    return matches[0]

# =========================================================
# 2. 입력 파일 찾기
# =========================================================
RAW_FILE = find_latest_file("bat_process.csv")
FINAL_FILE = find_latest_file("bat_process_outlier_final.csv")
SUMMARY_FILE = find_latest_file("outlier_final_summary.csv")
FLAG_FILE = find_latest_file("flag_summary.csv")

# 사용 파일 기록
used_files = pd.DataFrame({
    "role": ["RAW_FILE", "FINAL_FILE", "SUMMARY_FILE", "FLAG_FILE"],
    "path": [RAW_FILE, FINAL_FILE, SUMMARY_FILE, FLAG_FILE]
})
used_files.to_csv(os.path.join(LOG_DIR, "used_files.csv"), index=False, encoding="utf-8-sig")

print("사용 파일 경로")
print(used_files)

# =========================================================
# 3. 데이터 로드
# =========================================================
df_raw = pd.read_csv(RAW_FILE, encoding="euc-kr")
df_final = pd.read_csv(FINAL_FILE, encoding="utf-8-sig")
summary = pd.read_csv(SUMMARY_FILE, encoding="utf-8-sig")
flag_summary = pd.read_csv(FLAG_FILE, encoding="utf-8-sig")

if "judge" in df_final.columns:
    df_final["judge_bin"] = (df_final["judge"] == "불량").astype(int)

# =========================================================
# 4. 파일 기초 정보 저장
# =========================================================
meta_info = pd.DataFrame({
    "item": ["raw_rows", "raw_cols", "final_rows", "final_cols"],
    "value": [df_raw.shape[0], df_raw.shape[1], df_final.shape[0], df_final.shape[1]]
})
meta_info.to_csv(os.path.join(LOG_DIR, "meta_info.csv"), index=False, encoding="utf-8-sig")

# =========================================================
# 5. 시각화 1: 유형별 처리 규모 요약
# =========================================================
type_plot = (
    summary.groupby("type", as_index=False)
    .agg(
        variable_count=("column", "count"),
        total_changed=("changed_count", "sum"),
        mean_changed_ratio=("changed_ratio", "mean"),
        max_changed_ratio=("changed_ratio", "max")
    )
)

type_plot["mean_changed_ratio_pct"] = type_plot["mean_changed_ratio"] * 100
type_plot["max_changed_ratio_pct"] = type_plot["max_changed_ratio"] * 100
type_plot.to_csv(os.path.join(OUT_DIR, "01_type_summary_table.csv"), index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(type_plot["type"], type_plot["variable_count"])
axes[0].set_title("유형별 변수 개수")
axes[0].set_ylabel("Count")

axes[1].bar(type_plot["type"], type_plot["total_changed"])
axes[1].set_title("유형별 총 변경 건수")
axes[1].set_ylabel("Changed count")

axes[2].bar(type_plot["type"], type_plot["max_changed_ratio_pct"])
axes[2].set_title("유형별 최대 변경 비율")
axes[2].set_ylabel("Max changed ratio (%)")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "01_type_summary.png"))
plt.close()

# =========================================================
# 6. 시각화 2: B형 대표 변수 분포 + recipe centers
# =========================================================
b_candidates = ["c3_cvval", "c4_cvval", "c3_ccval", "c1_curr_end"]
b_candidates = [c for c in b_candidates if c in df_final.columns]

fig, axes = plt.subplots(len(b_candidates), 1, figsize=(8, 4 * max(1, len(b_candidates))))
if len(b_candidates) == 1:
    axes = [axes]

b_info_rows = []

for ax, col in zip(axes, b_candidates):
    ax.hist(df_final[col].dropna(), bins=40, alpha=0.8)

    row = summary[summary["column"] == col]
    centers = []
    if len(row) > 0:
        centers_text = str(row.iloc[0].get("recipe_centers", ""))
        if centers_text and centers_text != "nan":
            for x in centers_text.split(","):
                try:
                    centers.append(float(x))
                except:
                    pass

    for c in centers:
        ax.axvline(c, linestyle="--", linewidth=1.5)

    ax.set_title(f"{col} - B형 대표 분포와 recipe centers")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")

    changed_ratio = row.iloc[0]["changed_ratio"] * 100 if len(row) > 0 else np.nan
    b_info_rows.append({
        "column": col,
        "recipe_centers": ",".join(map(str, centers)),
        "changed_ratio_pct": changed_ratio
    })

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02_B_recipe_centers.png"))
plt.close()

pd.DataFrame(b_info_rows).to_csv(
    os.path.join(OUT_DIR, "02_B_recipe_centers_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 7. 시각화 3: C형 대표 변수 tail + flag별 불량률
# =========================================================
c_candidates = ["pg1_imp", "c4_time_cv"]
c_candidates = [c for c in c_candidates if c in df_final.columns]

fig, axes = plt.subplots(len(c_candidates), 2, figsize=(14, 5 * max(1, len(c_candidates))))
if len(c_candidates) == 1:
    axes = np.array([axes])

c_info_rows = []

for i, col in enumerate(c_candidates):
    row = summary[summary["column"] == col]
    p95 = row.iloc[0]["low"] if len(row) > 0 else np.nan
    p99 = row.iloc[0]["high"] if len(row) > 0 else np.nan

    # 왼쪽: 분포 + p95/p99
    axes[i, 0].hist(df_final[col].dropna(), bins=40, alpha=0.8)
    if pd.notna(p95):
        axes[i, 0].axvline(p95, linestyle="--", linewidth=1.5, label=f"p95={p95:.2f}")
    if pd.notna(p99):
        axes[i, 0].axvline(p99, linestyle="--", linewidth=1.5, label=f"p99={p99:.2f}")
    axes[i, 0].set_title(f"{col} - C형 tail 기준")
    axes[i, 0].legend()

    # 오른쪽: flag별 불량률
    flag_col = f"{col}_flag_p99"
    fail_rate_0, fail_rate_1 = np.nan, np.nan

    if flag_col in df_final.columns and "judge_bin" in df_final.columns:
        fail_rates = df_final.groupby(flag_col)["judge_bin"].mean()
        xs = [str(x) for x in fail_rates.index.tolist()]
        ys = fail_rates.values.tolist()
        axes[i, 1].bar(xs, ys)
        axes[i, 1].set_title(f"{col} - p99 flag별 불량률")
        axes[i, 1].set_xlabel("flag_p99")
        axes[i, 1].set_ylabel("Fail rate")

        if 0 in fail_rates.index:
            fail_rate_0 = fail_rates.loc[0]
        if 1 in fail_rates.index:
            fail_rate_1 = fail_rates.loc[1]
    else:
        axes[i, 1].text(0.5, 0.5, "judge_bin 또는 flag 없음", ha="center", va="center")
        axes[i, 1].set_title(f"{col} - flag별 불량률")

    c_info_rows.append({
        "column": col,
        "p95": p95,
        "p99": p99,
        "fail_rate_flag0": fail_rate_0,
        "fail_rate_flag1": fail_rate_1
    })

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "03_C_tail_and_failrate.png"))
plt.close()

pd.DataFrame(c_info_rows).to_csv(
    os.path.join(OUT_DIR, "03_C_tail_and_failrate_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 8. 시각화 4: D형 대표 변수 before/after 비교
# =========================================================
d_candidates = ["c3_time_cc", "c2_voltage_avg"]
d_candidates = [c for c in d_candidates if c in df_raw.columns and f"{c}_clip" in df_final.columns]

fig, axes = plt.subplots(len(d_candidates), 3, figsize=(16, 5 * max(1, len(d_candidates))))
if len(d_candidates) == 1:
    axes = np.array([axes])

d_info_rows = []

for i, col in enumerate(d_candidates):
    before = df_raw[col].dropna()
    after = df_final[f"{col}_clip"].dropna()

    # 전체 히스토그램
    axes[i, 0].hist(before, bins=40, alpha=0.6, label="Before")
    axes[i, 0].hist(after, bins=40, alpha=0.6, label="After")
    axes[i, 0].set_title(f"{col} - 전체 히스토그램")
    axes[i, 0].legend()

    # tail 확대
    all_s = pd.concat([before, after], axis=0)
    q95 = all_s.quantile(0.95)
    axes[i, 1].hist(before[before >= q95], bins=30, alpha=0.6, label="Before")
    axes[i, 1].hist(after[after >= q95], bins=30, alpha=0.6, label="After")
    axes[i, 1].set_title(f"{col} - 상위 5% tail")
    axes[i, 1].legend()

    # boxplot
    axes[i, 2].boxplot([before, after], labels=["Before", "After"])
    axes[i, 2].set_title(f"{col} - Boxplot")

    changed_row = summary[summary["column"] == col]
    changed_ratio = changed_row.iloc[0]["changed_ratio"] * 100 if len(changed_row) > 0 else np.nan

    d_info_rows.append({
        "column": col,
        "changed_ratio_pct": changed_ratio,
        "before_mean": before.mean(),
        "after_mean": after.mean(),
        "before_q95": before.quantile(0.95),
        "after_q95": after.quantile(0.95)
    })

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "04_D_before_after.png"))
plt.close()

pd.DataFrame(d_info_rows).to_csv(
    os.path.join(OUT_DIR, "04_D_before_after_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 9. 시각화 5: 상위 flag 비율
# =========================================================
top_flags = flag_summary.sort_values("flag_ratio", ascending=False).head(15)
top_flags.to_csv(os.path.join(OUT_DIR, "05_top_flag_ratios_table.csv"), index=False, encoding="utf-8-sig")

plt.figure(figsize=(10, 6))
plt.barh(top_flags["flag_col"], top_flags["flag_ratio_pct"])
plt.gca().invert_yaxis()
plt.xlabel("Flag ratio (%)")
plt.title("상위 15개 flag 비율")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "05_top_flag_ratios.png"))
plt.close()

# =========================================================
# 10. 자동 해석 메모 저장
# =========================================================
notes = []

notes.append("1. 유형별 처리 구조 요약: A형은 sanity check, B형은 recipe setpoint deviation, C형은 tail risk, D형은 분포 안정화 목적이다.")
notes.append("2. B형 대표 변수 그래프는 실제 setpoint 군집을 중심으로 기준이 설정되었는지 확인하기 위한 자료다.")
notes.append("3. C형 대표 변수 그래프는 극단값 flag가 실제로 불량률 차이를 만드는지 확인하기 위한 자료다.")
notes.append("4. D형 대표 변수 그래프는 clip이 전체 분포를 망가뜨리지 않고 tail만 정리했는지 확인하기 위한 자료다.")
notes.append("5. 상위 flag 비율 그래프는 어떤 위험 신호가 실제로 많이 발생했는지 빠르게 점검하기 위한 자료다.")

with open(os.path.join(LOG_DIR, "visual_interpretation_notes.txt"), "w", encoding="utf-8") as f:
    for line in notes:
        f.write(line + "\n")

# =========================================================
# 11. 완료 로그
# =========================================================
print("\n완료 ✅")
print(f"결과 폴더: {BASE_DIR}")
print(f"출력 그래프 폴더: {OUT_DIR}")
print(f"로그 폴더: {LOG_DIR}")
print("\n생성 파일")
print("- 01_type_summary.png / csv")
print("- 02_B_recipe_centers.png / csv")
print("- 03_C_tail_and_failrate.png / csv")
print("- 04_D_before_after.png / csv")
print("- 05_top_flag_ratios.png / csv")
print("- used_files.csv")
print("- visual_interpretation_notes.txt")

In [ ]:
import os
import glob
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False

# =========================================================
# 0. 실행 폴더 생성
# =========================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_DIR = f"outlier_meaning_check_{timestamp}"
OUT_DIR = os.path.join(BASE_DIR, "outputs")
LOG_DIR = os.path.join(BASE_DIR, "logs")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# =========================================================
# 1. 파일 자동 탐색
# =========================================================
def find_latest_file(filename_pattern: str) -> str:
    matches = glob.glob(f"**/{filename_pattern}", recursive=True)
    matches = [m for m in matches if os.path.isfile(m)]
    if len(matches) == 0:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {filename_pattern}")
    matches = sorted(matches, key=lambda x: os.path.getmtime(x), reverse=True)
    return matches[0]

RAW_FILE = find_latest_file("bat_process.csv")
FINAL_FILE = find_latest_file("bat_process_outlier_final.csv")
SUMMARY_FILE = find_latest_file("outlier_final_summary.csv")
FLAG_FILE = find_latest_file("flag_summary.csv")

used_files = pd.DataFrame({
    "role": ["RAW_FILE", "FINAL_FILE", "SUMMARY_FILE", "FLAG_FILE"],
    "path": [RAW_FILE, FINAL_FILE, SUMMARY_FILE, FLAG_FILE]
})
used_files.to_csv(os.path.join(LOG_DIR, "used_files.csv"), index=False, encoding="utf-8-sig")

print("사용 파일")
print(used_files)

# =========================================================
# 2. 데이터 로드
# =========================================================
df_raw = pd.read_csv(RAW_FILE, encoding="euc-kr")
df_final = pd.read_csv(FINAL_FILE, encoding="utf-8-sig")
summary = pd.read_csv(SUMMARY_FILE, encoding="utf-8-sig")
flag_summary = pd.read_csv(FLAG_FILE, encoding="utf-8-sig")

if "judge" not in df_final.columns:
    raise ValueError("judge 컬럼이 필요합니다.")

df_final["judge_bin"] = (df_final["judge"] == "불량").astype(int)
baseline_fail_rate = df_final["judge_bin"].mean()

# =========================================================
# 3. helper
# =========================================================
def safe_qcut(series, q=20):
    s = series.copy()
    try:
        return pd.qcut(s, q=q, duplicates="drop")
    except Exception:
        return pd.cut(s, bins=min(q, max(3, s.nunique())))

def quantile_compare_df(before, after, n=101):
    qs = np.linspace(0, 1, n)
    return pd.DataFrame({
        "q": qs,
        "before": before.quantile(qs).values,
        "after": after.quantile(qs).values
    })

# =========================================================
# 4. Plot 1: Flag uplift scatter
#    의미: "희귀하지만 강한 신호"인지, "흔하지만 무의미한 신호"인지 한눈에 확인
# =========================================================
flag_cols = [c for c in df_final.columns if c.endswith("_flag_recipe") or c.endswith("_flag_p95") or c.endswith("_flag_p99") or c.endswith("_flag_iqr")]

flag_rows = []
for col in flag_cols:
    prevalence = df_final[col].mean()
    if prevalence == 0:
        fail_flag0 = df_final.loc[df_final[col] == 0, "judge_bin"].mean()
        fail_flag1 = np.nan
        uplift = np.nan
    elif prevalence == 1:
        fail_flag0 = np.nan
        fail_flag1 = df_final.loc[df_final[col] == 1, "judge_bin"].mean()
        uplift = np.nan
    else:
        fail_flag0 = df_final.loc[df_final[col] == 0, "judge_bin"].mean()
        fail_flag1 = df_final.loc[df_final[col] == 1, "judge_bin"].mean()
        uplift = fail_flag1 - fail_flag0

    flag_rows.append({
        "flag_col": col,
        "prevalence": prevalence,
        "prevalence_pct": prevalence * 100,
        "fail_rate_flag0": fail_flag0,
        "fail_rate_flag1": fail_flag1,
        "uplift": uplift
    })

flag_uplift = pd.DataFrame(flag_rows)
flag_uplift.to_csv(os.path.join(OUT_DIR, "01_flag_uplift_table.csv"), index=False, encoding="utf-8-sig")

plot_df = flag_uplift.dropna(subset=["uplift"]).copy()
plot_df["type"] = np.where(plot_df["flag_col"].str.contains("_flag_recipe"), "B_recipe",
                   np.where(plot_df["flag_col"].str.contains("_flag_p"), "C_process", "D_normal"))

plt.figure(figsize=(10, 7))
for t, sub in plot_df.groupby("type"):
    plt.scatter(sub["prevalence_pct"], sub["uplift"], s=80, alpha=0.8, label=t)

# 상위 uplift 8개 라벨
label_df = plot_df.sort_values("uplift", ascending=False).head(8)
for _, r in label_df.iterrows():
    plt.annotate(r["flag_col"], (r["prevalence_pct"], r["uplift"]), fontsize=8, xytext=(4, 4), textcoords="offset points")

plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Flag prevalence (%)")
plt.ylabel("Fail-rate uplift (flag=1 - flag=0)")
plt.title("Flag의 의의: 발생비율 vs 불량률 상승")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "01_flag_uplift_scatter.png"))
plt.close()

# =========================================================
# 5. Plot 2: D형 clip 영향 - quantile comparison
#    의미: "중앙 분포는 유지되고 tail만 눌렸는가"
# =========================================================
d_top = summary[summary["type"] == "D_normal_variation"].sort_values("changed_ratio", ascending=False).head(2)["column"].tolist()

fig, axes = plt.subplots(len(d_top), 2, figsize=(12, 5 * max(1, len(d_top))))
if len(d_top) == 1:
    axes = np.array([axes])

d_quantile_rows = []

for i, col in enumerate(d_top):
    clip_col = f"{col}_clip"
    if clip_col not in df_final.columns or col not in df_raw.columns:
        continue

    before = df_raw[col].dropna()
    after = df_final[clip_col].dropna()

    qdf = quantile_compare_df(before, after, n=101)

    # 왼쪽: quantile-to-quantile
    axes[i, 0].plot(qdf["before"], qdf["after"], linewidth=2)
    minv = min(qdf["before"].min(), qdf["after"].min())
    maxv = max(qdf["before"].max(), qdf["after"].max())
    axes[i, 0].plot([minv, maxv], [minv, maxv], linestyle="--", linewidth=1)
    axes[i, 0].set_title(f"{col} - Quantile comparison")
    axes[i, 0].set_xlabel("Before quantiles")
    axes[i, 0].set_ylabel("After quantiles")

    # 오른쪽: q별 변화량
    axes[i, 1].plot(qdf["q"], qdf["after"] - qdf["before"], linewidth=2)
    axes[i, 1].axhline(0, linestyle="--", linewidth=1)
    axes[i, 1].set_title(f"{col} - Quantile shift")
    axes[i, 1].set_xlabel("Quantile")
    axes[i, 1].set_ylabel("After - Before")

    d_quantile_rows.append({
        "column": col,
        "before_q95": before.quantile(0.95),
        "after_q95": after.quantile(0.95),
        "before_q99": before.quantile(0.99),
        "after_q99": after.quantile(0.99),
        "changed_ratio_pct": summary.loc[summary["column"] == col, "changed_ratio"].iloc[0] * 100
    })

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02_D_quantile_effect.png"))
plt.close()

pd.DataFrame(d_quantile_rows).to_csv(
    os.path.join(OUT_DIR, "02_D_quantile_effect_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 6. Plot 3: B형 recipe center + flagged zone
#    의미: "평균이 아니라 실제 setpoint 중심으로 판정했다"를 보여줌
# =========================================================
b_top = summary[summary["type"] == "B_recipe_deviation"].sort_values("changed_ratio", ascending=False).head(3)["column"].tolist()

fig, axes = plt.subplots(len(b_top), 1, figsize=(10, 4 * max(1, len(b_top))))
if len(b_top) == 1:
    axes = [axes]

b_table_rows = []

for ax, col in zip(axes, b_top):
    s = df_final[col].dropna()
    ax.hist(s, bins=40, alpha=0.75)

    row = summary[summary["column"] == col].iloc[0]
    centers_text = str(row.get("recipe_centers", ""))
    centers = []
    if centers_text and centers_text != "nan":
        for x in centers_text.split(","):
            try:
                centers.append(float(x))
            except:
                pass

    # center 표시
    for c in centers:
        ax.axvline(c, linestyle="--", linewidth=1.5)

    # flag 비율
    flag_col = f"{col}_flag_recipe"
    flag_ratio = df_final[flag_col].mean() if flag_col in df_final.columns else np.nan

    ax.set_title(f"{col} - Recipe centers / flagged ratio={flag_ratio:.2%}")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")

    b_table_rows.append({
        "column": col,
        "recipe_centers": ",".join(map(str, centers)),
        "flag_ratio_pct": flag_ratio * 100 if pd.notna(flag_ratio) else np.nan
    })

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "03_B_recipe_center_story.png"))
plt.close()

pd.DataFrame(b_table_rows).to_csv(
    os.path.join(OUT_DIR, "03_B_recipe_center_story_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 7. Plot 4: C형 percentile별 불량률
#    의미: "tail이 실제로 위험해지는가"를 직접 보여줌
# =========================================================
c_top = summary[summary["type"] == "C_process_anomaly"].sort_values("changed_ratio", ascending=False).head(2)["column"].tolist()

fig, axes = plt.subplots(len(c_top), 1, figsize=(10, 4 * max(1, len(c_top))))
if len(c_top) == 1:
    axes = [axes]

c_bin_rows = []

for ax, col in zip(axes, c_top):
    tmp = df_final[[col, "judge_bin"]].dropna().copy()
    tmp["bin"] = safe_qcut(tmp[col], q=20)
    grp = tmp.groupby("bin", observed=False).agg(
        mean_x=(col, "mean"),
        fail_rate=("judge_bin", "mean"),
        n=(col, "size")
    ).reset_index(drop=True)

    ax.plot(range(len(grp)), grp["fail_rate"], marker="o")
    ax.set_title(f"{col} - Percentile bin별 불량률")
    ax.set_xlabel("Low → High percentile bins")
    ax.set_ylabel("Fail rate")
    ax.axhline(baseline_fail_rate, linestyle="--", linewidth=1)

    # tail uplift 저장
    c_bin_rows.append({
        "column": col,
        "baseline_fail_rate": baseline_fail_rate,
        "last_bin_fail_rate": grp["fail_rate"].iloc[-1],
        "first_bin_fail_rate": grp["fail_rate"].iloc[0],
        "last_minus_baseline": grp["fail_rate"].iloc[-1] - baseline_fail_rate
    })

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "04_C_percentile_failrate.png"))
plt.close()

pd.DataFrame(c_bin_rows).to_csv(
    os.path.join(OUT_DIR, "04_C_percentile_failrate_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 8. 해석 메모 저장
# =========================================================
notes = [
    "01_flag_uplift_scatter: 이상치 처리의 핵심 의의는 flag가 실제로 불량률을 높이는 위험군을 분리하는지 확인하는 것이다.",
    "02_D_quantile_effect: D형은 분포 전체를 바꾸는 것이 아니라 tail만 정리해야 하므로, 히스토그램보다 quantile 비교가 적절하다.",
    "03_B_recipe_center_story: B형은 중앙값이 아니라 실제 setpoint 군집 기준으로 판정했다는 점이 핵심이다.",
    "04_C_percentile_failrate: C형은 극단값 자체가 위험신호인지, 즉 값이 커질수록 불량률이 올라가는지를 보여주는 게 가장 중요하다."
]
with open(os.path.join(LOG_DIR, "meaning_notes.txt"), "w", encoding="utf-8") as f:
    for line in notes:
        f.write(line + "\n")

print("\n완료 ✅")
print(f"결과 폴더: {BASE_DIR}")
print(f"출력 그래프 폴더: {OUT_DIR}")
print(f"로그 폴더: {LOG_DIR}")
print("\n대표 그래프")
print("- 01_flag_uplift_scatter.png")
print("- 02_D_quantile_effect.png")
print("- 03_B_recipe_center_story.png")
print("- 04_C_percentile_failrate.png")

In [ ]:
import os
import glob
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False

# =========================================================
# 0. 실행 폴더 생성
# =========================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_DIR = f"outlier_final_visual_{timestamp}"
OUT_DIR = os.path.join(BASE_DIR, "outputs")
LOG_DIR = os.path.join(BASE_DIR, "logs")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# =========================================================
# 1. 최신 파일 자동 탐색
# =========================================================
def find_latest_file(filename_pattern: str) -> str:
    matches = glob.glob(f"**/{filename_pattern}", recursive=True)
    matches = [m for m in matches if os.path.isfile(m)]
    if len(matches) == 0:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {filename_pattern}")
    matches = sorted(matches, key=lambda x: os.path.getmtime(x), reverse=True)
    return matches[0]

RAW_FILE = find_latest_file("bat_process.csv")
FINAL_FILE = find_latest_file("bat_process_outlier_final.csv")
SUMMARY_FILE = find_latest_file("outlier_final_summary.csv")
FLAG_FILE = find_latest_file("flag_summary.csv")

used_files = pd.DataFrame({
    "role": ["RAW_FILE", "FINAL_FILE", "SUMMARY_FILE", "FLAG_FILE"],
    "path": [RAW_FILE, FINAL_FILE, SUMMARY_FILE, FLAG_FILE]
})
used_files.to_csv(os.path.join(LOG_DIR, "used_files.csv"), index=False, encoding="utf-8-sig")

# =========================================================
# 2. 데이터 로드
# =========================================================
df_raw = pd.read_csv(RAW_FILE, encoding="euc-kr")
df_final = pd.read_csv(FINAL_FILE, encoding="utf-8-sig")
summary = pd.read_csv(SUMMARY_FILE, encoding="utf-8-sig")
flag_summary = pd.read_csv(FLAG_FILE, encoding="utf-8-sig")

if "judge" not in df_final.columns:
    raise ValueError("judge 컬럼이 필요합니다.")

df_final["judge_bin"] = (df_final["judge"] == "불량").astype(int)
baseline_fail_rate = df_final["judge_bin"].mean()

# =========================================================
# 3. 대표 변수 선택
# =========================================================
# B형: 레시피 중심 검토용
b_cols = ["c3_cvval", "c4_cvval", "c3_ccval", "c1_curr_end"]
b_cols = [c for c in b_cols if c in df_final.columns]

# C형: 공정 이상 대표
c_cols = ["pg1_imp", "c4_time_cv"]
c_cols = [c for c in c_cols if c in df_final.columns]

# D형: 자연 변동 대표
d_cols = ["c3_time_cc", "c2_voltage_avg"]
d_cols = [c for c in d_cols if c in df_raw.columns and f"{c}_clip" in df_final.columns]

# =========================================================
# 4. 시각화 1: B형
#    히스토그램 + recipe centers
# =========================================================
fig, axes = plt.subplots(len(b_cols), 1, figsize=(9, 4 * max(1, len(b_cols))))
if len(b_cols) == 1:
    axes = [axes]

b_rows = []

for ax, col in zip(axes, b_cols):
    s = df_final[col].dropna()
    ax.hist(s, bins=40, alpha=0.8)

    row = summary[summary["column"] == col]
    centers = []
    changed_ratio = np.nan
    if len(row) > 0:
        changed_ratio = row.iloc[0]["changed_ratio"] * 100
        centers_text = str(row.iloc[0].get("recipe_centers", ""))
        if centers_text and centers_text != "nan":
            for x in centers_text.split(","):
                try:
                    centers.append(float(x))
                except:
                    pass

    for c in centers:
        ax.axvline(c, linestyle="--", linewidth=1.5)

    flag_col = f"{col}_flag_recipe"
    flag_ratio = df_final[flag_col].mean() * 100 if flag_col in df_final.columns else np.nan

    ax.set_title(f"{col} | flagged={flag_ratio:.2f}% | changed={changed_ratio:.2f}%")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")

    b_rows.append({
        "column": col,
        "recipe_centers": ",".join(map(str, centers)),
        "flag_ratio_pct": flag_ratio,
        "changed_ratio_pct": changed_ratio
    })

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "01_B_recipe_hist.png"))
plt.close()

pd.DataFrame(b_rows).to_csv(
    os.path.join(OUT_DIR, "01_B_recipe_hist_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 5. 시각화 2: C형
#    (왼쪽) 히스토그램 + p95/p99
#    (오른쪽) 양품/불량 boxplot
# =========================================================
fig, axes = plt.subplots(len(c_cols), 2, figsize=(14, 5 * max(1, len(c_cols))))
if len(c_cols) == 1:
    axes = np.array([axes])

c_rows = []

for i, col in enumerate(c_cols):
    row = summary[summary["column"] == col]
    p95 = row.iloc[0]["low"] if len(row) > 0 else np.nan
    p99 = row.iloc[0]["high"] if len(row) > 0 else np.nan

    # 히스토그램 + threshold
    s = df_final[col].dropna()
    axes[i, 0].hist(s, bins=40, alpha=0.8)
    if pd.notna(p95):
        axes[i, 0].axvline(p95, linestyle="--", linewidth=1.5, label=f"p95={p95:.2f}")
    if pd.notna(p99):
        axes[i, 0].axvline(p99, linestyle="--", linewidth=1.5, label=f"p99={p99:.2f}")
    axes[i, 0].set_title(f"{col} - histogram with tail thresholds")
    axes[i, 0].legend()

    # 양품/불량 boxplot
    good = df_final.loc[df_final["judge_bin"] == 0, col].dropna()
    bad = df_final.loc[df_final["judge_bin"] == 1, col].dropna()
    axes[i, 1].boxplot([good, bad], labels=["양품", "불량"])
    axes[i, 1].set_title(f"{col} - 양품/불량 boxplot")

    flag_col = f"{col}_flag_p99"
    fail_rate_0, fail_rate_1 = np.nan, np.nan
    if flag_col in df_final.columns:
        if (df_final[flag_col] == 0).sum() > 0:
            fail_rate_0 = df_final.loc[df_final[flag_col] == 0, "judge_bin"].mean()
        if (df_final[flag_col] == 1).sum() > 0:
            fail_rate_1 = df_final.loc[df_final[flag_col] == 1, "judge_bin"].mean()

    c_rows.append({
        "column": col,
        "p95": p95,
        "p99": p99,
        "fail_rate_flag0": fail_rate_0,
        "fail_rate_flag1": fail_rate_1
    })

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02_C_hist_box.png"))
plt.close()

pd.DataFrame(c_rows).to_csv(
    os.path.join(OUT_DIR, "02_C_hist_box_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 6. 시각화 3: D형
#    before/after 히스토그램 + before/after 박스플롯
# =========================================================
fig, axes = plt.subplots(len(d_cols), 2, figsize=(14, 5 * max(1, len(d_cols))))
if len(d_cols) == 1:
    axes = np.array([axes])

d_rows = []

for i, col in enumerate(d_cols):
    before = df_raw[col].dropna()
    after = df_final[f"{col}_clip"].dropna()

    # 히스토그램
    axes[i, 0].hist(before, bins=40, alpha=0.6, label="Before")
    axes[i, 0].hist(after, bins=40, alpha=0.6, label="After")
    axes[i, 0].set_title(f"{col} - before/after histogram")
    axes[i, 0].legend()

    # 박스플롯
    axes[i, 1].boxplot([before, after], labels=["Before", "After"])
    axes[i, 1].set_title(f"{col} - before/after boxplot")

    changed_row = summary[summary["column"] == col]
    changed_ratio = changed_row.iloc[0]["changed_ratio"] * 100 if len(changed_row) > 0 else np.nan

    d_rows.append({
        "column": col,
        "changed_ratio_pct": changed_ratio,
        "before_q95": before.quantile(0.95),
        "after_q95": after.quantile(0.95),
        "before_q99": before.quantile(0.99),
        "after_q99": after.quantile(0.99)
    })

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "03_D_hist_box_before_after.png"))
plt.close()

pd.DataFrame(d_rows).to_csv(
    os.path.join(OUT_DIR, "03_D_hist_box_before_after_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 7. 시각화 4: flag별 불량률 비교
# =========================================================
flag_cols = [c for c in df_final.columns if c.endswith("_flag_recipe") or c.endswith("_flag_p99") or c.endswith("_flag_iqr")]

rows = []
for col in flag_cols:
    if df_final[col].nunique() < 2:
        continue

    fail0 = df_final.loc[df_final[col] == 0, "judge_bin"].mean()
    fail1 = df_final.loc[df_final[col] == 1, "judge_bin"].mean()
    rows.append({
        "flag_col": col,
        "fail_rate_flag0": fail0,
        "fail_rate_flag1": fail1,
        "uplift": fail1 - fail0,
        "prevalence_pct": df_final[col].mean() * 100
    })

flag_effect = pd.DataFrame(rows).sort_values("uplift", ascending=False)
flag_effect.to_csv(
    os.path.join(OUT_DIR, "04_flag_failrate_table.csv"),
    index=False, encoding="utf-8-sig"
)

top_flag_effect = flag_effect.head(12)

plt.figure(figsize=(10, 6))
plt.barh(top_flag_effect["flag_col"], top_flag_effect["uplift"])
plt.gca().invert_yaxis()
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Fail-rate uplift (flag=1 - flag=0)")
plt.title("flag별 불량률 상승 효과")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "04_flag_failrate_uplift.png"))
plt.close()

# =========================================================
# 8. 최종 해석 메모
# =========================================================
notes = []
notes.append("B형 해석: 히스토그램에서 값이 특정 setpoint 군집 주변에 모이고, recipe centers가 그 중심을 통과하면 기준 설정이 타당하다고 본다.")
notes.append("C형 해석: 히스토그램의 오른쪽 tail에 p95/p99 선이 위치하고, boxplot에서 불량군이 상방으로 치우치면 공정 이상 신호로 해석한다.")
notes.append("D형 해석: before/after 히스토그램과 박스플롯에서 중앙부는 크게 유지되고 꼬리만 정리되면 적절한 clip으로 본다.")
notes.append("flag 해석: flag=1의 불량률이 flag=0보다 높을수록 이상치 처리가 실제 위험군 분리에 기여한 것으로 본다.")

with open(os.path.join(LOG_DIR, "final_interpretation_notes.txt"), "w", encoding="utf-8") as f:
    for line in notes:
        f.write(line + "\n")

print("\n완료 ✅")
print(f"결과 폴더: {BASE_DIR}")
print(f"출력 그래프 폴더: {OUT_DIR}")
print(f"로그 폴더: {LOG_DIR}")
print("\n대표 그래프")
print("- 01_B_recipe_hist.png")
print("- 02_C_hist_box.png")
print("- 03_D_hist_box_before_after.png")
print("- 04_flag_failrate_uplift.png")

In [ ]:
import os
import glob
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 0. 기본 설정
# =========================================================
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_DIR = f"outlier_full_run_{timestamp}"
DATA_DIR = os.path.join(BASE_DIR, "data")
TABLE_DIR = os.path.join(BASE_DIR, "tables")
PLOT_DIR = os.path.join(BASE_DIR, "plots")
LOG_DIR = os.path.join(BASE_DIR, "logs")

for d in [BASE_DIR, DATA_DIR, TABLE_DIR, PLOT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# =========================================================
# 1. 입력 파일 자동 탐색
#    - bat_process.csv만 있으면 동작
# =========================================================
def find_latest_file(filename_pattern: str) -> str:
    matches = glob.glob(f"**/{filename_pattern}", recursive=True)
    matches = [m for m in matches if os.path.isfile(m)]
    if len(matches) == 0:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {filename_pattern}")
    matches = sorted(matches, key=lambda x: os.path.getmtime(x), reverse=True)
    return matches[0]

RAW_FILE = find_latest_file("bat_process.csv")

used_files = pd.DataFrame({
    "role": ["RAW_FILE"],
    "path": [RAW_FILE]
})
used_files.to_csv(os.path.join(LOG_DIR, "used_files.csv"), index=False, encoding="utf-8-sig")

# =========================================================
# 2. 데이터 로드
# =========================================================
df = pd.read_csv(RAW_FILE, encoding="euc-kr")
df_out = df.copy()

if "judge" not in df_out.columns:
    raise ValueError("judge 컬럼이 필요합니다.")

df_out["judge_bin"] = (df_out["judge"] == "불량").astype(int)
n_rows = len(df_out)
baseline_fail_rate = df_out["judge_bin"].mean()

# =========================================================
# 3. 변수 그룹 정의
# =========================================================
A_DATA_ERROR = [
    "ocv1_ocv", "ocv2_ocv", "socv1_ocv", "socv2_ocv", "socv3_ocv",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "pg1_imp", "pg1_impfit", "pc1_imp", "m1_res_ac",
    "m1_thick",
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
]

B_RECIPE = [
    "c1_curr_end", "dc1_curr_end", "c2_curr_end", "dc2_curr_end",
    "c3_curr_end", "dc3_curr_end", "c4_curr_end",
    "c3_cvval", "c4_cvval",
    "c3_ccval", "c4_ccval",
]

C_PROCESS = [
    "ocv2_deltaocv", "pg1_imp", "pc1_imp", "m1_res_ac", "m1_thick",
    "c3_time_cv", "c4_time_cv",
]

D_NORMAL = [
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "c1_voltage_avg", "dc1_voltage_avg",
    "c2_voltage_avg", "dc2_voltage_avg",
    "c3_voltage_avg", "dc3_voltage_avg",
    "c4_voltage_avg",
]

# 존재하는 컬럼만 사용
A_DATA_ERROR = [c for c in A_DATA_ERROR if c in df_out.columns]
B_RECIPE = [c for c in B_RECIPE if c in df_out.columns]
C_PROCESS = [c for c in C_PROCESS if c in df_out.columns]
D_NORMAL = [c for c in D_NORMAL if c in df_out.columns]

# =========================================================
# 4. B형 변수별 설정
# =========================================================
B_CONFIG = {
    "c3_cvval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": 20, "tol_ratio": None},
    "c4_cvval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": 20, "tol_ratio": None},

    "c1_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "dc1_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "c2_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "dc2_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "c3_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "dc3_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "c4_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},

    "c3_ccval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.03},
    "c4_ccval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.03},
}

# =========================================================
# 5. 유틸 함수
# =========================================================
def apply_hard_limit(series, low=None, high=None):
    s = series.copy()
    if low is not None:
        s = s.mask(s < low, np.nan)
    if high is not None:
        s = s.mask(s > high, np.nan)
    return s

def iqr_bounds(series, k=1.5):
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    return low, high

def iqr_clip(series, k=1.5):
    low, high = iqr_bounds(series, k=k)
    return series.clip(lower=low, upper=high), low, high

def get_main_recipe_centers(series, top_k=3, min_ratio=0.05, round_digits=None):
    s = series.dropna().copy()
    if len(s) == 0:
        return []
    if round_digits is not None:
        s = s.round(round_digits)
    vc = s.value_counts(normalize=True)
    centers = vc[vc >= min_ratio].index.tolist()[:top_k]
    if len(centers) == 0:
        centers = vc.index.tolist()[:1]
    return sorted(centers)

def make_recipe_flag(series, centers, tol_abs=None, tol_ratio=None):
    s = series.copy()
    if len(centers) == 0:
        return pd.Series(0, index=s.index, dtype=int)

    ok_mask = pd.Series(False, index=s.index)
    for c in centers:
        tol = tol_abs if tol_abs is not None else abs(c) * tol_ratio
        ok_mask = ok_mask | ((s >= c - tol) & (s <= c + tol))
    return (~ok_mask).astype(int)

def safe_qcut(series, q=20):
    s = series.copy()
    try:
        return pd.qcut(s, q=q, duplicates="drop")
    except Exception:
        return pd.cut(s, bins=min(q, max(3, s.nunique())))

# =========================================================
# 6. A형 처리: sanity check
# =========================================================
summary_rows = []

A_LIMITS = {}
for col in A_DATA_ERROR:
    if "temp" in col:
        A_LIMITS[col] = (0, 1000)
    elif "time" in col:
        A_LIMITS[col] = (0, 100000)
    elif "ocv" in col or "voltage" in col:
        A_LIMITS[col] = (0, 5000)
    elif "imp" in col or "res" in col:
        A_LIMITS[col] = (0, 10000)
    elif "thick" in col:
        A_LIMITS[col] = (0, 10000)
    else:
        A_LIMITS[col] = (None, None)

for col in A_DATA_ERROR:
    before = df_out[col].copy()
    low, high = A_LIMITS[col]
    after = apply_hard_limit(before, low=low, high=high)
    df_out[col] = after

    changed = before.notna().sum() - after.notna().sum()

    summary_rows.append({
        "column": col,
        "type": "A_data_error",
        "action": "hard_limit_to_nan",
        "low": low,
        "high": high,
        "changed_count": int(changed),
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

# =========================================================
# 7. B형 처리: recipe center 기반
# =========================================================
for col in B_RECIPE:
    s = df_out[col].dropna()
    if len(s) == 0:
        continue

    cfg = B_CONFIG.get(col, {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.03})

    centers = get_main_recipe_centers(
        s,
        top_k=cfg["top_k"],
        min_ratio=cfg["min_ratio"],
        round_digits=cfg["round_digits"]
    )

    flag = make_recipe_flag(
        series=df_out[col],
        centers=centers,
        tol_abs=cfg["tol_abs"],
        tol_ratio=cfg["tol_ratio"]
    )

    flag_col = f"{col}_flag_recipe"
    df_out[flag_col] = flag

    changed = int(df_out[flag_col].sum())

    if cfg["tol_abs"] is not None:
        low_display = min(centers) - cfg["tol_abs"] if len(centers) else np.nan
        high_display = max(centers) + cfg["tol_abs"] if len(centers) else np.nan
    else:
        low_display = min(centers) * (1 - cfg["tol_ratio"]) if len(centers) else np.nan
        high_display = max(centers) * (1 + cfg["tol_ratio"]) if len(centers) else np.nan

    summary_rows.append({
        "column": col,
        "type": "B_recipe_deviation",
        "action": "flag_only_mode_centers",
        "low": low_display,
        "high": high_display,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ",".join(map(str, centers))
    })

# =========================================================
# 8. C형 처리: 실제 p95/p99 계산 후 flag
# =========================================================
for col in C_PROCESS:
    s = df_out[col].dropna()
    if len(s) == 0:
        continue

    p95 = s.quantile(0.95)
    p99 = s.quantile(0.99)

    df_out[f"{col}_flag_p95"] = (df_out[col] > p95).astype(int)
    df_out[f"{col}_flag_p99"] = (df_out[col] > p99).astype(int)

    changed = int(df_out[f"{col}_flag_p99"].sum())

    summary_rows.append({
        "column": col,
        "type": "C_process_anomaly",
        "action": "flag_only_p95_p99",
        "low": p95,
        "high": p99,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

# =========================================================
# 9. D형 처리: IQR clip + flag
# =========================================================
for col in D_NORMAL:
    before = df_out[col].copy()
    clipped, low, high = iqr_clip(before, k=1.5)

    df_out[f"{col}_clip"] = clipped
    df_out[f"{col}_flag_iqr"] = ((before < low) | (before > high)).astype(int)

    changed = int(df_out[f"{col}_flag_iqr"].sum())

    summary_rows.append({
        "column": col,
        "type": "D_normal_variation",
        "action": "iqr_clip_and_flag",
        "low": low,
        "high": high,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

# =========================================================
# 10. 결과 저장
# =========================================================
summary = pd.DataFrame(summary_rows).sort_values(["type", "changed_ratio"], ascending=[True, False])
summary.to_csv(os.path.join(TABLE_DIR, "outlier_final_summary.csv"), index=False, encoding="utf-8-sig")
df_out.to_csv(os.path.join(DATA_DIR, "bat_process_outlier_final.csv"), index=False, encoding="utf-8-sig")

type_summary = (
    summary.groupby(["type", "action"], as_index=False)
    .agg(
        variable_count=("column", "count"),
        total_changed=("changed_count", "sum"),
        mean_changed_ratio=("changed_ratio", "mean"),
        max_changed_ratio=("changed_ratio", "max")
    )
)
type_summary["mean_changed_ratio_pct"] = type_summary["mean_changed_ratio"] * 100
type_summary["max_changed_ratio_pct"] = type_summary["max_changed_ratio"] * 100
type_summary.to_csv(os.path.join(TABLE_DIR, "type_summary.csv"), index=False, encoding="utf-8-sig")

flag_cols = [
    c for c in df_out.columns
    if c.endswith("_flag_recipe") or c.endswith("_flag_p95") or c.endswith("_flag_p99") or c.endswith("_flag_iqr")
]

flag_summary = pd.DataFrame({
    "flag_col": flag_cols,
    "flag_count": [df_out[c].sum() for c in flag_cols],
    "flag_ratio": [df_out[c].mean() for c in flag_cols]
}).sort_values("flag_ratio", ascending=False)
flag_summary["flag_ratio_pct"] = flag_summary["flag_ratio"] * 100
flag_summary.to_csv(os.path.join(TABLE_DIR, "flag_summary.csv"), index=False, encoding="utf-8-sig")

# =========================================================
# 11. 대표 시각화 1: B형 히스토그램 + recipe centers
# =========================================================
b_cols = ["c3_cvval", "c4_cvval", "c3_ccval", "c1_curr_end"]
b_cols = [c for c in b_cols if c in df_out.columns]

fig, axes = plt.subplots(len(b_cols), 1, figsize=(9, 4 * max(1, len(b_cols))))
if len(b_cols) == 1:
    axes = [axes]

b_rows = []

for ax, col in zip(axes, b_cols):
    s = df_out[col].dropna()
    ax.hist(s, bins=40, alpha=0.8)

    row = summary[summary["column"] == col]
    centers = []
    changed_ratio = np.nan
    if len(row) > 0:
        changed_ratio = row.iloc[0]["changed_ratio"] * 100
        centers_text = str(row.iloc[0].get("recipe_centers", ""))
        if centers_text and centers_text != "nan":
            for x in centers_text.split(","):
                try:
                    centers.append(float(x))
                except:
                    pass

    for c in centers:
        ax.axvline(c, linestyle="--", linewidth=1.5)

    flag_col = f"{col}_flag_recipe"
    flag_ratio = df_out[flag_col].mean() * 100 if flag_col in df_out.columns else np.nan

    ax.set_title(f"{col} | flagged={flag_ratio:.2f}% | changed={changed_ratio:.2f}%")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")

    b_rows.append({
        "column": col,
        "recipe_centers": ",".join(map(str, centers)),
        "flag_ratio_pct": flag_ratio,
        "changed_ratio_pct": changed_ratio
    })

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_B_recipe_hist.png"))
plt.close()

pd.DataFrame(b_rows).to_csv(
    os.path.join(TABLE_DIR, "01_B_recipe_hist_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 12. 대표 시각화 2: C형 히스토그램 + 양품/불량 boxplot
#     (실제 p95/p99를 바로 다시 계산해서 그림)
# =========================================================
c_cols = ["pg1_imp", "c4_time_cv"]
c_cols = [c for c in c_cols if c in df_out.columns]

fig, axes = plt.subplots(len(c_cols), 2, figsize=(14, 5 * max(1, len(c_cols))))
if len(c_cols) == 1:
    axes = np.array([axes])

c_rows = []

for i, col in enumerate(c_cols):
    s = df_out[col].dropna()
    p95 = s.quantile(0.95)
    p99 = s.quantile(0.99)

    axes[i, 0].hist(s, bins=40, alpha=0.8)
    axes[i, 0].axvline(p95, linestyle="--", linewidth=1.5, label=f"p95={p95:.2f}")
    axes[i, 0].axvline(p99, linestyle="--", linewidth=1.5, label=f"p99={p99:.2f}")
    axes[i, 0].set_title(f"{col} - histogram with actual p95/p99")
    axes[i, 0].legend()

    good = df_out.loc[df_out["judge_bin"] == 0, col].dropna()
    bad = df_out.loc[df_out["judge_bin"] == 1, col].dropna()
    axes[i, 1].boxplot([good, bad], labels=["양품", "불량"])
    axes[i, 1].set_title(f"{col} - 양품/불량 boxplot")

    flag_col = f"{col}_flag_p99"
    fail_rate_0, fail_rate_1 = np.nan, np.nan
    if flag_col in df_out.columns:
        if (df_out[flag_col] == 0).sum() > 0:
            fail_rate_0 = df_out.loc[df_out[flag_col] == 0, "judge_bin"].mean()
        if (df_out[flag_col] == 1).sum() > 0:
            fail_rate_1 = df_out.loc[df_out[flag_col] == 1, "judge_bin"].mean()

    c_rows.append({
        "column": col,
        "p95_actual": p95,
        "p99_actual": p99,
        "fail_rate_flag0": fail_rate_0,
        "fail_rate_flag1": fail_rate_1
    })

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_C_hist_box.png"))
plt.close()

pd.DataFrame(c_rows).to_csv(
    os.path.join(TABLE_DIR, "02_C_hist_box_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 13. 대표 시각화 3: D형 before/after 히스토그램 + boxplot
# =========================================================
d_cols = ["c3_time_cc", "c2_voltage_avg"]
d_cols = [c for c in d_cols if c in df.columns and f"{c}_clip" in df_out.columns]

fig, axes = plt.subplots(len(d_cols), 2, figsize=(14, 5 * max(1, len(d_cols))))
if len(d_cols) == 1:
    axes = np.array([axes])

d_rows = []

for i, col in enumerate(d_cols):
    before = df[col].dropna()
    after = df_out[f"{col}_clip"].dropna()

    axes[i, 0].hist(before, bins=40, alpha=0.6, label="Before")
    axes[i, 0].hist(after, bins=40, alpha=0.6, label="After")
    axes[i, 0].set_title(f"{col} - before/after histogram")
    axes[i, 0].legend()

    axes[i, 1].boxplot([before, after], labels=["Before", "After"])
    axes[i, 1].set_title(f"{col} - before/after boxplot")

    changed_row = summary[summary["column"] == col]
    changed_ratio = changed_row.iloc[0]["changed_ratio"] * 100 if len(changed_row) > 0 else np.nan

    d_rows.append({
        "column": col,
        "changed_ratio_pct": changed_ratio,
        "before_q95": before.quantile(0.95),
        "after_q95": after.quantile(0.95),
        "before_q99": before.quantile(0.99),
        "after_q99": after.quantile(0.99)
    })

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_D_hist_box_before_after.png"))
plt.close()

pd.DataFrame(d_rows).to_csv(
    os.path.join(TABLE_DIR, "03_D_hist_box_before_after_table.csv"),
    index=False, encoding="utf-8-sig"
)

# =========================================================
# 14. 대표 시각화 4: flag별 불량률 uplift
# =========================================================
rows = []
for col in flag_cols:
    if df_out[col].nunique() < 2:
        continue

    fail0 = df_out.loc[df_out[col] == 0, "judge_bin"].mean()
    fail1 = df_out.loc[df_out[col] == 1, "judge_bin"].mean()

    rows.append({
        "flag_col": col,
        "fail_rate_flag0": fail0,
        "fail_rate_flag1": fail1,
        "uplift": fail1 - fail0,
        "prevalence_pct": df_out[col].mean() * 100
    })

flag_effect = pd.DataFrame(rows).sort_values("uplift", ascending=False)
flag_effect.to_csv(
    os.path.join(TABLE_DIR, "04_flag_failrate_table.csv"),
    index=False, encoding="utf-8-sig"
)

top_flag_effect = flag_effect.head(12)

plt.figure(figsize=(10, 6))
plt.barh(top_flag_effect["flag_col"], top_flag_effect["uplift"])
plt.gca().invert_yaxis()
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Fail-rate uplift (flag=1 - flag=0)")
plt.title("flag별 불량률 상승 효과")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_flag_failrate_uplift.png"))
plt.close()

# =========================================================
# 15. 해석 메모 자동 저장
# =========================================================
notes = []
notes.append("A형: 데이터 파손 여부만 확인하는 sanity check 단계다. 변경이 0건이어도 정상이다.")
notes.append("B형: 레시피형 변수는 평균이 아니라 실제 setpoint 군집 중심으로 판정해야 하므로 recipe centers 기반 flag를 사용했다.")
notes.append("C형: 공정 이상형 변수는 extreme value 자체가 불량 신호일 수 있으므로 삭제 대신 p95/p99 기반 flag로 보존했다.")
notes.append("D형: 자연 변동형 변수는 전체 분포를 유지하면서 tail만 완만하게 정리하기 위해 IQR clip + flag를 함께 사용했다.")
notes.append("B형 그래프는 기준이 실제 setpoint 군집을 통과하는지 확인하는 자료다.")
notes.append("C형 그래프는 값이 커질수록 불량군으로 치우치는지 확인하는 자료다.")
notes.append("D형 그래프는 clip이 중심 분포를 유지하고 tail만 정리했는지 확인하는 자료다.")
notes.append("Flag uplift 그래프는 이상치 처리가 실제 위험군 분리에 기여했는지를 정량적으로 보여주는 핵심 자료다.")

with open(os.path.join(LOG_DIR, "final_interpretation_notes.txt"), "w", encoding="utf-8") as f:
    for line in notes:
        f.write(line + "\n")

# =========================================================
# 16. 실행 요약 저장
# =========================================================
meta_info = pd.DataFrame({
    "item": ["raw_rows", "raw_cols", "final_rows", "final_cols", "baseline_fail_rate"],
    "value": [df.shape[0], df.shape[1], df_out.shape[0], df_out.shape[1], baseline_fail_rate]
})
meta_info.to_csv(os.path.join(LOG_DIR, "meta_info.csv"), index=False, encoding="utf-8-sig")

print("\n완료 ✅")
print(f"결과 폴더: {BASE_DIR}")
print(f"데이터 폴더: {DATA_DIR}")
print(f"표 폴더: {TABLE_DIR}")
print(f"그래프 폴더: {PLOT_DIR}")
print(f"로그 폴더: {LOG_DIR}")
print("\n핵심 결과 파일")
print("- data/bat_process_outlier_final.csv")
print("- tables/outlier_final_summary.csv")
print("- tables/type_summary.csv")
print("- tables/flag_summary.csv")
print("- plots/01_B_recipe_hist.png")
print("- plots/02_C_hist_box.png")
print("- plots/03_D_hist_box_before_after.png")
print("- plots/04_flag_failrate_uplift.png")